# Lab Day 2 — DeepWeeds / Kaggle
Notebook tự chứa code, theo README/GUIDE/RUBRIC gốc (không sửa ba tài liệu và eval.py).

1. Bật **GPU T4 ×2** và Internet; Add Input Dataset có `data/images` và `data/labels`.
2. Lần đầu: giữ `STAGE="prepare"`, **Run All** để EDA, kiểm tra split và overfit batch nhỏ. Dừng xem kết quả.
3. Đổi `STAGE="screen"`, **Run All** để chạy B01–B05, T00–T10 và I00/I01/I02/I03/I08/I07 trên val.
4. Duyệt bảng validation, điền `SELECTION_REASON`, đổi `STAGE="final"`, Run All: khóa cấu hình rồi chạy T00/F01 với 3 seed trên test.
5. `STAGE="report"` chỉ đọc các kết quả đã lưu, xuất workbook/eval/ảnh/ZIP.

Mọi lần Run All đều khởi tạo các biến từ đầu. Kết quả hoàn tất được tái sử dụng khi config/source/split và artifact khớp.
Để thử riêng B03: `STAGE="backbones"`, `RUN_BACKBONES_B01_B02=False`, `BACKBONE_FILTER=["B03"]`, rồi Run All.
**Import .ipynb không khôi phục biến Python hoặc checkpoint.** Phiên mới cần gắn `day2_output` cũ làm Input và điền resume.
Theo dõi log epoch và RAM/VRAM mỗi phút; không có progress widget `train batches`.


## Dataset và lưu tiến độ
Upload từ local thành Kaggle Dataset, giữ nguyên ảnh/CSV gốc. Cấu trúc `data/images/*.jpg`,
`data/labels/{labels.csv,train_subset0.csv,val_subset0.csv,test_subset0.csv}`; cũng chấp nhận `images.zip`.
Notebook tự tìm ở cả đường dẫn `/kaggle/input/<slug>` và `/kaggle/input/datasets/<owner>/<slug>`.
Không cần cột Species trong các split CSV. Mọi kiểm tra đủ 17.509 ảnh, nhãn và giao rỗng vẫn bắt buộc.

Output nằm trong `/kaggle/working/day2_output`. Checkpoint được lưu mỗi epoch.
Muốn tiếp tục ở phiên mới: lưu/download **output folder**, không chỉ tải notebook.
`day2_submission.zip` chỉ có sản phẩm nhỏ; cần toàn bộ `day2_output` để resume.


In [ ]:
# Giới hạn tài nguyên TRƯỚC import torch; worker mới cũng dùng các giới hạn này.
import os
for key, value in {"OMP_NUM_THREADS":"1", "MKL_NUM_THREADS":"1", "OPENBLAS_NUM_THREADS":"1",
                   "MALLOC_ARENA_MAX":"2", "TORCH_CUDNN_V8_API_LRU_CACHE_LIMIT":"128",
                   "HF_HUB_DISABLE_PROGRESS_BARS":"1"}.items():
    os.environ[key] = value
import importlib.util
import subprocess
import sys

required = {"timm": "timm==1.0.30", "openpyxl": "openpyxl", "matplotlib": "matplotlib", "tqdm": "tqdm"}
missing = [package for module, package in required.items()
           if importlib.util.find_spec(module) is None]
if missing:
    print(f"Cài thư viện còn thiếu: {missing}", flush=True)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
else:
    print("Các thư viện notebook cần thiết đã có sẵn.", flush=True)

from importlib.metadata import version
if version("timm") != "1.0.30":
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "timm==1.0.30"])


In [ ]:
from pathlib import Path
import json
import os
import shutil
import sys
import zipfile
import torch

WORK = Path("/kaggle/working")
INPUT_ROOT = Path("/kaggle/input")
CODE_DIR = WORK / "day2_code"
OUTPUT_ROOT = WORK / "day2_output"
RUNS_DIR = OUTPUT_ROOT / "runs"
PRED_DIR = OUTPUT_ROOT / "predictions"
CURVES_DIR = OUTPUT_ROOT / "curves"
EVAL_DIR = OUTPUT_ROOT / "eval"
# Lần chạy mới: để trống cả hai. Resume: điền đúng một output cũ dưới /kaggle/input.
RESUME_OUTPUT_DIR = ""  # e.g. /kaggle/input/day2-output/day2_output
RESUME_ARCHIVE = ""     # e.g. /kaggle/input/day2-output/day2_output.zip
PACKAGE_RESUME_OUTPUT = False  # True creates a zip containing checkpoints; it can be large
if RESUME_OUTPUT_DIR and RESUME_ARCHIVE:
    raise ValueError("Chỉ điền một trong RESUME_OUTPUT_DIR hoặc RESUME_ARCHIVE; lần chạy mới hãy để cả hai trống.")

def resume_input_path(value, setting):
    path = Path(value).resolve()
    input_root = INPUT_ROOT.resolve()
    if input_root not in path.parents:
        raise ValueError(
            f"{setting} phải trỏ tới output cũ đã gắn dưới {INPUT_ROOT}. "
            "Lần chạy mới hãy để cả RESUME_OUTPUT_DIR và RESUME_ARCHIVE trống."
        )
    return path

has_working_runs = ((OUTPUT_ROOT / "experiment_records.json").exists()
                    or any((OUTPUT_ROOT / "runs").rglob("checkpoint*.pt")))
if (RESUME_OUTPUT_DIR or RESUME_ARCHIVE) and has_working_runs:
    print("Working đã có tiến độ; bỏ qua restore để không ghi đè bằng output cũ.",flush=True)
elif RESUME_OUTPUT_DIR:
    resume_dir = resume_input_path(RESUME_OUTPUT_DIR, "RESUME_OUTPUT_DIR")
    if not resume_dir.is_dir():
        raise FileNotFoundError(f"Không tìm thấy thư mục output cũ đã gắn: {resume_dir}")
    shutil.copytree(resume_dir, OUTPUT_ROOT, dirs_exist_ok=True)
elif RESUME_ARCHIVE:
    resume_archive = resume_input_path(RESUME_ARCHIVE, "RESUME_ARCHIVE")
    if not resume_archive.is_file():
        raise FileNotFoundError(f"Không tìm thấy file archive output cũ đã gắn: {resume_archive}")
    with zipfile.ZipFile(resume_archive) as archive:
        root = WORK.resolve()
        for member in archive.infolist():
            target = (WORK / member.filename).resolve()
            if target != root and root not in target.parents:
                raise ValueError(f"Unsafe path in resume archive: {member.filename}")
        archive.extractall(WORK)
for folder in (CODE_DIR, OUTPUT_ROOT, RUNS_DIR, PRED_DIR, CURVES_DIR, EVAL_DIR):
    folder.mkdir(parents=True, exist_ok=True)

# Phần slug trong đường dẫn Dataset dưới /kaggle/input; để trống nếu chỉ có một bộ phù hợp.
DATASET_SLUG = ""
EPOCHS = 12
BATCH_SIZE = 32
FINAL_SEEDS = [0, 1, 2]
USE_DATA_PARALLEL = True
B03_USE_DATA_PARALLEL = False  # Thử riêng ConvNeXt trên 1 T4 sau khi B03 liên tục làm kernel chết.
NUM_WORKERS = 0  # Nạp ảnh trong tiến trình chính để giảm CPU/RAM trên Kaggle.
# prepare -> screen -> final. Các stage riêng giúp chia phiên theo hạn mức GPU.
STAGE = "prepare"  # prepare | screen | backbones | training | inference | final | report
RUN_BACKBONES_B01_B02 = True
RUN_BACKBONES_B03_B05 = True
BACKBONE_FILTER = []  # [] = đủ nhóm; ["B03"] để chạy riêng ConvNeXt sau khi setup.
SELECTION_REASON = ""  # Trước final: ghi lý do chọn dựa vào F1 val và latency thực đo.
RUN_BACKBONES = STAGE in {"screen", "backbones"}
RUN_ABLATIONS = STAGE in {"screen", "training"}
RUN_INFERENCE = STAGE in {"screen", "inference"}
RUN_FINAL = STAGE == "final"
if STAGE not in {"prepare","screen","backbones","training","inference","final","report"}:
    raise ValueError("STAGE không hợp lệ")
if (OUTPUT_ROOT / "final_selection.json").exists() and STAGE not in {"final","report"}:
    raise RuntimeError("Đã khóa cấu hình chung kết; dùng STAGE='final' hoặc 'report', không chọn lại bằng test.")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_num_threads(1)
print(f"Python={sys.version.split()[0]} | torch={torch.__version__} | device={DEVICE}", flush=True)
print(f"CPU threads={torch.get_num_threads()} | DataLoader workers={NUM_WORKERS}", flush=True)
print(f"B03 DataParallel={B03_USE_DATA_PARALLEL} (False dùng một GPU T4)", flush=True)
if DEVICE.type == "cuda":
    GPU_COUNT = torch.cuda.device_count()
    GPU_NAMES = [torch.cuda.get_device_name(i) for i in range(GPU_COUNT)]
    DATA_PARALLEL_ACTIVE = bool(USE_DATA_PARALLEL and GPU_COUNT > 1)
    print(f"GPU count={GPU_COUNT} | GPUs={GPU_NAMES}", flush=True)
    print(f"DataParallel requested={USE_DATA_PARALLEL} | active for training={DATA_PARALLEL_ACTIVE}", flush=True)
else:
    GPU_COUNT, GPU_NAMES, DATA_PARALLEL_ACTIVE = 0, [], False
    print("Chưa gắn GPU: mở Settings → Accelerator → GPU trước khi train.", flush=True)
    raise RuntimeError("Notebook yêu cầu Kaggle GPU để chạy thí nghiệm và đo latency.")


## Code và kiểm tra phiên bản
Cell sau giải mã đúng byte của các module đã review; cell kế tiếp kiểm tra SHA256 trước khi import.
Source dễ đọc và chỉnh nằm trong `/kaggle/working/day2_code/*.py`, được đưa vào ZIP sản phẩm.


In [ ]:
# Embedded source: generated by code/sync_notebook.py
import base64
EMBEDDED_SOURCE = {
  "runtime.py": "IiIiQm91bmQgaG9zdCByZXNvdXJjZXMgYmVmb3JlIGltcG9ydGluZyB0b3JjaDsgZGlhZ25vc3RpY3MgbmV2ZXIgcmV0YWluIHRlbnNvcnMuIiIiCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBqc29uCmltcG9ydCBvcwpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKCgpkZWYgY29uZmlndXJlX2Vudmlyb25tZW50KCk6CiAgICAjIGN1RE5OIGRlZmF1bHRzIHRvIHVwIHRvIDEwLDAwMCBjYWNoZWQgcGxhbnMgKH4yIEdpQikuIEtlZXAgYSBib3VuZGVkIGNhY2hlLgogICAgIyBodHRwczovL2RvY3MucHl0b3JjaC5vcmcvZG9jcy9zdGFibGUvY3VkYV9lbnZpcm9ubWVudF92YXJpYWJsZXMuaHRtbAogICAgZGVmYXVsdHMgPSB7Ik9NUF9OVU1fVEhSRUFEUyI6ICIxIiwgIk1LTF9OVU1fVEhSRUFEUyI6ICIxIiwKICAgICAgICAgICAgICAgICJPUEVOQkxBU19OVU1fVEhSRUFEUyI6ICIxIiwgIk5VTUVYUFJfTlVNX1RIUkVBRFMiOiAiMSIsCiAgICAgICAgICAgICAgICAiTUFMTE9DX0FSRU5BX01BWCI6ICIyIiwgIlRPUkNIX0NVRE5OX1Y4X0FQSV9MUlVfQ0FDSEVfTElNSVQiOiAiMTI4IiwKICAgICAgICAgICAgICAgICJUT0tFTklaRVJTX1BBUkFMTEVMSVNNIjogImZhbHNlIiwgIkhGX0hVQl9ESVNBQkxFX1BST0dSRVNTX0JBUlMiOiAiMSJ9CiAgICBmb3Iga2V5LCB2YWx1ZSBpbiBkZWZhdWx0cy5pdGVtcygpOgogICAgICAgIG9zLmVudmlyb24uc2V0ZGVmYXVsdChrZXksIHZhbHVlKQoKCmRlZiBhdG9taWNfanNvbihwYXRoLCB2YWx1ZSk6CiAgICBwYXRoID0gUGF0aChwYXRoKQogICAgcGF0aC5wYXJlbnQubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgdGVtcCA9IHBhdGgud2l0aF9zdWZmaXgocGF0aC5zdWZmaXggKyAiLnRtcCIpCiAgICB0ZW1wLndyaXRlX3RleHQoanNvbi5kdW1wcyh2YWx1ZSwgaW5kZW50PTIsIGVuc3VyZV9hc2NpaT1GYWxzZSwgYWxsb3dfbmFuPUZhbHNlKSwgZW5jb2Rpbmc9InV0Zi04IikKICAgIHRlbXAucmVwbGFjZShwYXRoKQoKCmRlZiBtZW1vcnlfc25hcHNob3QoKToKICAgICIiIkxpbnV4IHByb2Nlc3MgUlNTIGFuZCBjZ3JvdXAgaGVhZHJvb20gKHJlY2xhaW1hYmxlIGZpbGUgY2FjaGUgZXhjbHVkZWQpLiIiIgogICAgcmVzdWx0ID0ge30KICAgIHRyeToKICAgICAgICByb3dzID0gUGF0aCgiL3Byb2Mvc2VsZi9zdGF0dXMiKS5yZWFkX3RleHQoKS5zcGxpdGxpbmVzKCkKICAgICAgICByZXN1bHRbInJzc19naWIiXSA9IG5leHQoaW50KHIuc3BsaXQoKVsxXSkgLyAxMDI0KioyIGZvciByIGluIHJvd3MgaWYgci5zdGFydHN3aXRoKCJWbVJTUzoiKSkKICAgIGV4Y2VwdCAoT1NFcnJvciwgU3RvcEl0ZXJhdGlvbiwgVmFsdWVFcnJvcik6CiAgICAgICAgcGFzcwogICAgcm9vdCA9IFBhdGgoIi9zeXMvZnMvY2dyb3VwIikKICAgIGZvciB1c2VkLCBsaW1pdCwgc3RhdCwgY2FjaGVfa2V5IGluICgKICAgICAgICAocm9vdCAvICJtZW1vcnkuY3VycmVudCIsIHJvb3QgLyAibWVtb3J5Lm1heCIsIHJvb3QgLyAibWVtb3J5LnN0YXQiLCAiaW5hY3RpdmVfZmlsZSIpLAogICAgICAgIChyb290IC8gIm1lbW9yeS9tZW1vcnkudXNhZ2VfaW5fYnl0ZXMiLCByb290IC8gIm1lbW9yeS9tZW1vcnkubGltaXRfaW5fYnl0ZXMiLAogICAgICAgICByb290IC8gIm1lbW9yeS9tZW1vcnkuc3RhdCIsICJ0b3RhbF9pbmFjdGl2ZV9maWxlIiksCiAgICApOgogICAgICAgIHRyeToKICAgICAgICAgICAgY3VycmVudCwgbWF4aW11bSA9IGludCh1c2VkLnJlYWRfdGV4dCgpKSwgaW50KGxpbWl0LnJlYWRfdGV4dCgpKQogICAgICAgICAgICBpZiBtYXhpbXVtID49IDIqKjYwOgogICAgICAgICAgICAgICAgY29udGludWUKICAgICAgICAgICAgc3RhdHMgPSBkaWN0KGxpbmUuc3BsaXQoKSBmb3IgbGluZSBpbiBzdGF0LnJlYWRfdGV4dCgpLnNwbGl0bGluZXMoKSkKICAgICAgICAgICAgcmVjbGFpbWFibGUgPSBpbnQoc3RhdHMuZ2V0KGNhY2hlX2tleSwgMCkpCiAgICAgICAgICAgIHJlc3VsdC51cGRhdGUoY2dyb3VwX2dpYj1jdXJyZW50IC8gMTAyNCoqMywgbGltaXRfZ2liPW1heGltdW0gLyAxMDI0KiozLAogICAgICAgICAgICAgICAgICAgICAgICAgIGhlYWRyb29tX2dpYj0obWF4aW11bSAtIGN1cnJlbnQgKyByZWNsYWltYWJsZSkgLyAxMDI0KiozKQogICAgICAgICAgICBicmVhawogICAgICAgIGV4Y2VwdCAoT1NFcnJvciwgVmFsdWVFcnJvcik6CiAgICAgICAgICAgIHBhc3MKICAgIHJldHVybiByZXN1bHQKCgpkZWYgY2hlY2tfbWVtb3J5KG1heF9yc3NfZ2liPTE4LjAsIG1pbl9oZWFkcm9vbV9naWI9My4wKToKICAgIHN0YXRzID0gbWVtb3J5X3NuYXBzaG90KCkKICAgIGlmICgobWF4X3Jzc19naWIgYW5kIHN0YXRzLmdldCgicnNzX2dpYiIsIDApID49IG1heF9yc3NfZ2liKQogICAgICAgICAgICBvciBzdGF0cy5nZXQoImhlYWRyb29tX2dpYiIsIGZsb2F0KCJpbmYiKSkgPCBtaW5faGVhZHJvb21fZ2liKToKICAgICAgICByYWlzZSBNZW1vcnlFcnJvcihmIkhvc3QgUkFNIGd1YXJkIHN0b3BwZWQgdGhpcyBleHBlcmltZW50OiB7c3RhdHN9LiAiCiAgICAgICAgICAgICAgICAgICAgICAgICAgIlRoZSBsYXN0IGNvbXBsZXRlZCBlcG9jaCByZW1haW5zIGluIGNoZWNrcG9pbnRfbGFzdC5wdDsgc2VlIHJ1bi5sb2cuIikKICAgIHJldHVybiBzdGF0cwoKCmRlZiByZXNvdXJjZV9yZXBvcnQoKToKICAgIGltcG9ydCB0b3JjaAogICAgcGFydHMgPSBbZiJ7a2V5fT17dmFsdWU6LjJmfSIgZm9yIGtleSwgdmFsdWUgaW4gbWVtb3J5X3NuYXBzaG90KCkuaXRlbXMoKV0KICAgIGlmIHRvcmNoLmN1ZGEuaXNfYXZhaWxhYmxlKCk6CiAgICAgICAgZm9yIGkgaW4gcmFuZ2UodG9yY2guY3VkYS5kZXZpY2VfY291bnQoKSk6CiAgICAgICAgICAgIHBhcnRzLmFwcGVuZChmImdwdXtpfV9hbGxvYz17dG9yY2guY3VkYS5tZW1vcnlfYWxsb2NhdGVkKGkpLzEwMjQqKjM6LjJmfUdpQiAiCiAgICAgICAgICAgICAgICAgICAgICAgICBmInJlc2VydmVkPXt0b3JjaC5jdWRhLm1lbW9yeV9yZXNlcnZlZChpKS8xMDI0KiozOi4yZn1HaUIiKQogICAgcmV0dXJuICIgfCAiLmpvaW4ocGFydHMpIG9yICJSQU0gc3RhdHMgdW5hdmFpbGFibGUgb24gdGhpcyBPUyIK",
  "dataset.py": "IiIiRGVlcFdlZWRzIGxvYWRpbmcsIHNwbGl0IHZhbGlkYXRpb24sIHRyYW5zZm9ybXMsIGFuZCBEYXRhTG9hZGVyIGhlbHBlcnMuIiIiCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCByYW5kb20KZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCgppbXBvcnQgcGFuZGFzIGFzIHBkCgp0cnk6CiAgICBmcm9tIHRvcmNoLnV0aWxzLmRhdGEgaW1wb3J0IERhdGFzZXQgYXMgX1RvcmNoRGF0YXNldApleGNlcHQgSW1wb3J0RXJyb3I6ICAjIGFsbG93cyBtZXRhZGF0YS1vbmx5IHRvb2xzIHRvIGltcG9ydCB0aGUgbW9kdWxlIG91dHNpZGUgS2FnZ2xlCiAgICBjbGFzcyBfVG9yY2hEYXRhc2V0OgogICAgICAgIHBhc3MKCk5VTV9DTEFTU0VTID0gOQpFWFBFQ1RFRF9JTUFHRVMgPSAxN181MDkKQ0xBU1NfTkFNRVMgPSBbCiAgICAiQ2hpbmVlIEFwcGxlIiwgIkxhbnRhbmEiLCAiUGFya2luc29uaWEiLCAiUGFydGhlbml1bSIsICJQcmlja2x5IEFjYWNpYSIsCiAgICAiUnViYmVyIFZpbmUiLCAiU2lhbSBXZWVkIiwgIlNuYWtlIFdlZWQiLCAiTmVnYXRpdmVzIiwKXQpJTUFHRU5FVF9NRUFOID0gKDAuNDg1LCAwLjQ1NiwgMC40MDYpCklNQUdFTkVUX1NURCA9ICgwLjIyOSwgMC4yMjQsIDAuMjI1KQoKCmRlZiBsb2FkX3NwbGl0KGxhYmVsc19kaXI6IHN0ciB8IFBhdGgsIGZvbGQ6IGludCA9IDApOgogICAgIiIiUmVhZCB0cmFpbi92YWwvdGVzdCBDU1ZzOyBzcGxpdCBtZXRhZGF0YSBvbmx5IG5lZWRzIEZpbGVuYW1lIGFuZCBMYWJlbC4iIiIKICAgIGlmIG5vdCAwIDw9IGludChmb2xkKSA8PSA0OgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoImZvbGQgbXVzdCBiZSBiZXR3ZWVuIDAgYW5kIDQiKQogICAgbGFiZWxzX2RpciA9IFBhdGgobGFiZWxzX2RpcikKICAgIGZyYW1lcyA9IFtdCiAgICBmb3Igc3BsaXQgaW4gKCJ0cmFpbiIsICJ2YWwiLCAidGVzdCIpOgogICAgICAgIHBhdGggPSBsYWJlbHNfZGlyIC8gZiJ7c3BsaXR9X3N1YnNldHtpbnQoZm9sZCl9LmNzdiIKICAgICAgICBpZiBub3QgcGF0aC5pc19maWxlKCk6CiAgICAgICAgICAgIHJhaXNlIEZpbGVOb3RGb3VuZEVycm9yKGYiTWlzc2luZyBzcGxpdCBDU1Y6IHtwYXRofSIpCiAgICAgICAgZnJhbWUgPSBwZC5yZWFkX2NzdihwYXRoKQogICAgICAgIG1pc3NpbmcgPSB7IkZpbGVuYW1lIiwgIkxhYmVsIn0gLSBzZXQoZnJhbWUuY29sdW1ucykKICAgICAgICBpZiBtaXNzaW5nOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYie3BhdGh9IGlzIG1pc3NpbmcgY29sdW1uczoge3NvcnRlZChtaXNzaW5nKX0iKQogICAgICAgIGZyYW1lcy5hcHBlbmQoZnJhbWUpCiAgICByZXR1cm4gdHVwbGUoZnJhbWVzKQoKCmRlZiBjaGVja19zcGxpdCh0cmFpbl9kZjogcGQuRGF0YUZyYW1lLCB2YWxfZGY6IHBkLkRhdGFGcmFtZSwgdGVzdF9kZjogcGQuRGF0YUZyYW1lLAogICAgICAgICAgICAgICAgaW1hZ2VzX2Rpcjogc3RyIHwgUGF0aCwgdmVyYm9zZTogYm9vbCA9IFRydWUpIC0+IGRpY3Q6CiAgICAiIiJWYWxpZGF0ZSBmb2xkIGludGVncml0eSBhbmQgaW1hZ2UgYXZhaWxhYmlsaXR5OyByZXR1cm4gY291bnRzIGZvciB0aGUgcmVwb3J0LiIiIgogICAgaW1hZ2VzX2RpciA9IFBhdGgoaW1hZ2VzX2RpcikKICAgIGZyYW1lcyA9IHsidHJhaW4iOiB0cmFpbl9kZiwgInZhbCI6IHZhbF9kZiwgInRlc3QiOiB0ZXN0X2RmfQogICAgbmFtZXMgPSB7fQogICAgY291bnRzID0ge30KICAgIHBlcl9jbGFzcyA9IHt9CiAgICBmb3Igc3BsaXQsIGZyYW1lIGluIGZyYW1lcy5pdGVtcygpOgogICAgICAgIG1pc3NpbmdfY29scyA9IHsiRmlsZW5hbWUiLCAiTGFiZWwifSAtIHNldChmcmFtZS5jb2x1bW5zKQogICAgICAgIGlmIG1pc3NpbmdfY29sczoKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIntzcGxpdH0gQ1NWIGlzIG1pc3NpbmcgY29sdW1uczoge3NvcnRlZChtaXNzaW5nX2NvbHMpfSIpCiAgICAgICAgaWYgZnJhbWVbIkZpbGVuYW1lIl0uaXNuYSgpLmFueSgpIG9yIGZyYW1lWyJMYWJlbCJdLmlzbmEoKS5hbnkoKToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIntzcGxpdH0gQ1NWIGhhcyBlbXB0eSBGaWxlbmFtZSBvciBMYWJlbCB2YWx1ZXMiKQogICAgICAgIGZpbGVuYW1lcyA9IGZyYW1lWyJGaWxlbmFtZSJdLmFzdHlwZShzdHIpCiAgICAgICAgZHVwbGljYXRlcyA9IGZpbGVuYW1lc1tmaWxlbmFtZXMuZHVwbGljYXRlZCgpXS51bmlxdWUoKS50b2xpc3QoKQogICAgICAgIGlmIGR1cGxpY2F0ZXM6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJ7c3BsaXR9IGNvbnRhaW5zIGR1cGxpY2F0ZSBmaWxlbmFtZXM6IHtkdXBsaWNhdGVzWzo1XX0iKQogICAgICAgIG51bWVyaWNfbGFiZWxzID0gcGQudG9fbnVtZXJpYyhmcmFtZVsiTGFiZWwiXSwgZXJyb3JzPSJyYWlzZSIpCiAgICAgICAgaWYgKG5vdCBudW1lcmljX2xhYmVscy5tb2QoMSkuZXEoMCkuYWxsKCkKICAgICAgICAgICAgICAgIG9yIG5vdCBudW1lcmljX2xhYmVscy5iZXR3ZWVuKDAsIE5VTV9DTEFTU0VTIC0gMSkuYWxsKCkpOgogICAgICAgICAgICBpbnZhbGlkID0gbnVtZXJpY19sYWJlbHNbfm51bWVyaWNfbGFiZWxzLmJldHdlZW4oMCwgTlVNX0NMQVNTRVMgLSAxKQogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgfCB+bnVtZXJpY19sYWJlbHMubW9kKDEpLmVxKDApXS51bmlxdWUoKS50b2xpc3QoKQogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYie3NwbGl0fSBoYXMgaW52YWxpZCBsYWJlbHMgb3V0c2lkZSBpbnRlZ2VyIHJhbmdlIDAuLntOVU1fQ0xBU1NFUyAtIDF9OiB7aW52YWxpZFs6NV19IikKICAgICAgICBsYWJlbHMgPSBudW1lcmljX2xhYmVscy5hc3R5cGUoaW50KQogICAgICAgIG5hbWVzW3NwbGl0XSA9IHNldChmaWxlbmFtZXMpCiAgICAgICAgY291bnRzW3NwbGl0XSA9IGludChsZW4oZnJhbWUpKQogICAgICAgIGNsYXNzX2NvdW50cyA9IGxhYmVscy52YWx1ZV9jb3VudHMoKS5yZWluZGV4KHJhbmdlKE5VTV9DTEFTU0VTKSwgZmlsbF92YWx1ZT0wKQogICAgICAgIHBlcl9jbGFzc1tzcGxpdF0gPSB7Q0xBU1NfTkFNRVNbaV06IGludChjbGFzc19jb3VudHMuaWxvY1tpXSkgZm9yIGkgaW4gcmFuZ2UoTlVNX0NMQVNTRVMpfQoKICAgIG92ZXJsYXBzID0gewogICAgICAgICJ0cmFpbl92YWwiOiBzb3J0ZWQobmFtZXNbInRyYWluIl0gJiBuYW1lc1sidmFsIl0pLAogICAgICAgICJ0cmFpbl90ZXN0Ijogc29ydGVkKG5hbWVzWyJ0cmFpbiJdICYgbmFtZXNbInRlc3QiXSksCiAgICAgICAgInZhbF90ZXN0Ijogc29ydGVkKG5hbWVzWyJ2YWwiXSAmIG5hbWVzWyJ0ZXN0Il0pLAogICAgfQogICAgaWYgYW55KG92ZXJsYXBzLnZhbHVlcygpKToKICAgICAgICBub25lbXB0eSA9IHtrZXk6IHZhbHVlWzo1XSBmb3Iga2V5LCB2YWx1ZSBpbiBvdmVybGFwcy5pdGVtcygpIGlmIHZhbHVlfQogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJGb2xkIGhhcyBvdmVybGFwcGluZyBmaWxlbmFtZXM6IHtub25lbXB0eX0iKQogICAgdG90YWwgPSBsZW4obmFtZXNbInRyYWluIl0gfCBuYW1lc1sidmFsIl0gfCBuYW1lc1sidGVzdCJdKQogICAgaWYgdG90YWwgIT0gRVhQRUNURURfSU1BR0VTOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJGb2xkIHVuaW9uIGhhcyB7dG90YWx9IGltYWdlczsgZXhwZWN0ZWQge0VYUEVDVEVEX0lNQUdFU30iKQogICAgbWlzc2luZ19pbWFnZXMgPSBbbmFtZSBmb3IgbmFtZSBpbiBuYW1lc1sidHJhaW4iXSB8IG5hbWVzWyJ2YWwiXSB8IG5hbWVzWyJ0ZXN0Il0KICAgICAgICAgICAgICAgICAgICAgIGlmIG5vdCAoaW1hZ2VzX2RpciAvIG5hbWUpLmlzX2ZpbGUoKV0KICAgIGlmIG1pc3NpbmdfaW1hZ2VzOgogICAgICAgIHJhaXNlIEZpbGVOb3RGb3VuZEVycm9yKAogICAgICAgICAgICBmIntsZW4obWlzc2luZ19pbWFnZXMpfSBDU1YgaW1hZ2VzIGFyZSBhYnNlbnQgZnJvbSB7aW1hZ2VzX2Rpcn07IGV4YW1wbGVzOiB7bWlzc2luZ19pbWFnZXNbOjVdfSIKICAgICAgICApCgogICAgcmF0aW9zID0ge2tleTogY291bnRzW2tleV0gLyB0b3RhbCBmb3Iga2V5IGluIGNvdW50c30KICAgIGlmIGFueShhYnMocmF0aW9zW2tleV0gLSB0YXJnZXQpID4gMC4wMSBmb3Iga2V5LCB0YXJnZXQgaW4KICAgICAgICAgICAoKCJ0cmFpbiIsIDAuNiksICgidmFsIiwgMC4yKSwgKCJ0ZXN0IiwgMC4yKSkpOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJTcGxpdCBwcm9wb3J0aW9ucyBkaWZmZXIgYnkgbW9yZSB0aGFuIDEgcGVyY2VudGFnZSBwb2ludDoge3JhdGlvc30iKQogICAgaWYgdmVyYm9zZToKICAgICAgICBwcmludChmIlNwbGl0IGNvdW50czoge2NvdW50c307IHVuaW9uPXt0b3RhbH07IG92ZXJsYXBzPTAiKQogICAgICAgIGZvciBzcGxpdCwgdmFsdWVzIGluIHBlcl9jbGFzcy5pdGVtcygpOgogICAgICAgICAgICBwcmludChmIntzcGxpdH0gY2xhc3MgY291bnRzOiB7dmFsdWVzfSIpCiAgICByZXR1cm4geyJuIjogY291bnRzLCAicGVyX2NsYXNzIjogcGVyX2NsYXNzLCAib3ZlcmxhcCI6IG92ZXJsYXBzLAogICAgICAgICAgICAidW5pb24iOiB0b3RhbCwgInJhdGlvcyI6IHJhdGlvc30KCgpkZWYgYnVpbGRfdHJhbnNmb3Jtcyh0cmFpbjogYm9vbCwgaW1nX3NpemU6IGludCA9IDIyNCwgYXVnOiBzdHIgPSAiYmFzaWMiKToKICAgICIiIkJ1aWxkIGRldGVybWluaXN0aWMgdmFsaWRhdGlvbiB0cmFuc2Zvcm1zIG9yIG9uZSBvZiB0aGUgZG9jdW1lbnRlZCB0cmFpbiBwb2xpY2llcy4iIiIKICAgIGZyb20gdG9yY2h2aXNpb24gaW1wb3J0IHRyYW5zZm9ybXMKCiAgICBpZiBpbWdfc2l6ZSA8IDMyOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoImltZ19zaXplIG11c3QgYmUgYXQgbGVhc3QgMzIiKQogICAgaWYgdHJhaW46CiAgICAgICAgb3BzID0gW3RyYW5zZm9ybXMuUmFuZG9tUmVzaXplZENyb3AoaW1nX3NpemUpLCB0cmFuc2Zvcm1zLlJhbmRvbUhvcml6b250YWxGbGlwKCldCiAgICAgICAgaWYgYXVnID09ICJjb2xvciI6CiAgICAgICAgICAgIG9wcy5hcHBlbmQodHJhbnNmb3Jtcy5Db2xvckppdHRlcigwLjIsIDAuMiwgMC4yLCAwLjA1KSkKICAgICAgICBlbGlmIGF1ZyA9PSAidHJpdmlhbCI6CiAgICAgICAgICAgIGF1Z21lbnQgPSBnZXRhdHRyKHRyYW5zZm9ybXMsICJUcml2aWFsQXVnbWVudFdpZGUiLCBOb25lKQogICAgICAgICAgICBvcHMuYXBwZW5kKGF1Z21lbnQoKSBpZiBhdWdtZW50IGVsc2UgdHJhbnNmb3Jtcy5SYW5kQXVnbWVudCgpKQogICAgICAgIGVsaWYgYXVnID09ICJyYW5kYXVnIjoKICAgICAgICAgICAgb3BzLmFwcGVuZCh0cmFuc2Zvcm1zLlJhbmRBdWdtZW50KG51bV9vcHM9MiwgbWFnbml0dWRlPTkpKQogICAgICAgIGVsaWYgYXVnICE9ICJiYXNpYyI6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJVbmtub3duIGF1Z21lbnRhdGlvbjoge2F1Z30iKQogICAgZWxzZToKICAgICAgICAjIERlZXBXZWVkcyBpbWFnZXMgYXJlIDI1NngyNTYuIFJlc2l6ZSB0aGVuIGNlbnRlciBjcm9wLCB3aXRoIG5vIHJhbmRvbSBvcHMuCiAgICAgICAgb3BzID0gW3RyYW5zZm9ybXMuUmVzaXplKGltZ19zaXplICsgMzIpLCB0cmFuc2Zvcm1zLkNlbnRlckNyb3AoaW1nX3NpemUpXQogICAgb3BzLmV4dGVuZChbdHJhbnNmb3Jtcy5Ub1RlbnNvcigpLCB0cmFuc2Zvcm1zLk5vcm1hbGl6ZShJTUFHRU5FVF9NRUFOLCBJTUFHRU5FVF9TVEQpXSkKICAgIHJldHVybiB0cmFuc2Zvcm1zLkNvbXBvc2Uob3BzKQoKCmNsYXNzIERlZXBXZWVkc0RhdGFzZXQoX1RvcmNoRGF0YXNldCk6CiAgICAiIiJTbWFsbCBpbi1tZW1vcnkgaW5kZXggb3ZlciBpbWFnZSBwYXRoczsgaW1hZ2UgcGl4ZWxzIGFyZSBkZWNvZGVkIG9uIGRlbWFuZC4iIiIKCiAgICBkZWYgX19pbml0X18oc2VsZiwgZGY6IHBkLkRhdGFGcmFtZSwgaW1hZ2VzX2Rpcjogc3RyIHwgUGF0aCwgdHJhbnNmb3JtPU5vbmUpOgogICAgICAgIGlmIG5vdCBpc2luc3RhbmNlKGRmLCBwZC5EYXRhRnJhbWUpOgogICAgICAgICAgICByYWlzZSBUeXBlRXJyb3IoImRmIG11c3QgYmUgYSBwYW5kYXMgRGF0YUZyYW1lIikKICAgICAgICBpZiBub3QgeyJGaWxlbmFtZSIsICJMYWJlbCJ9Lmlzc3Vic2V0KGRmLmNvbHVtbnMpOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJkZiBtdXN0IGNvbnRhaW4gRmlsZW5hbWUgYW5kIExhYmVsIGNvbHVtbnMiKQogICAgICAgIHNlbGYuZmlsZW5hbWVzID0gZGZbIkZpbGVuYW1lIl0uYXN0eXBlKHN0cikudG9saXN0KCkKICAgICAgICBudW1lcmljX2xhYmVscyA9IHBkLnRvX251bWVyaWMoZGZbIkxhYmVsIl0sIGVycm9ycz0icmFpc2UiKQogICAgICAgIGlmIChudW1lcmljX2xhYmVscy5pc25hKCkuYW55KCkgb3Igbm90IG51bWVyaWNfbGFiZWxzLm1vZCgxKS5lcSgwKS5hbGwoKQogICAgICAgICAgICAgICAgb3Igbm90IG51bWVyaWNfbGFiZWxzLmJldHdlZW4oMCwgTlVNX0NMQVNTRVMgLSAxKS5hbGwoKSk6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJMYWJlbCB2YWx1ZXMgbXVzdCBiZSBpbnRlZ2VycyBpbiAwLi57TlVNX0NMQVNTRVMgLSAxfSIpCiAgICAgICAgc2VsZi5sYWJlbHMgPSBudW1lcmljX2xhYmVscy5hc3R5cGUoaW50KS50b2xpc3QoKQogICAgICAgIHNlbGYuaW1hZ2VzX2RpciA9IFBhdGgoaW1hZ2VzX2RpcikKICAgICAgICBzZWxmLnRyYW5zZm9ybSA9IHRyYW5zZm9ybQoKICAgIGRlZiBfX2xlbl9fKHNlbGYpIC0+IGludDoKICAgICAgICByZXR1cm4gbGVuKHNlbGYuZmlsZW5hbWVzKQoKICAgIGRlZiBfX2dldGl0ZW1fXyhzZWxmLCBpOiBpbnQpOgogICAgICAgIGZyb20gUElMIGltcG9ydCBJbWFnZQoKICAgICAgICBwYXRoID0gc2VsZi5pbWFnZXNfZGlyIC8gc2VsZi5maWxlbmFtZXNbaV0KICAgICAgICB3aXRoIEltYWdlLm9wZW4ocGF0aCkgYXMgaW1hZ2U6CiAgICAgICAgICAgIGltYWdlID0gaW1hZ2UuY29udmVydCgiUkdCIikKICAgICAgICAgICAgaWYgc2VsZi50cmFuc2Zvcm0gaXMgbm90IE5vbmU6CiAgICAgICAgICAgICAgICBpbWFnZSA9IHNlbGYudHJhbnNmb3JtKGltYWdlKQogICAgICAgIHJldHVybiBpbWFnZSwgaW50KHNlbGYubGFiZWxzW2ldKSwgc2VsZi5maWxlbmFtZXNbaV0KCgpkZWYgc2VlZF93b3JrZXIod29ya2VyX2lkKToKICAgIGltcG9ydCBudW1weSBhcyBucAogICAgaW1wb3J0IHRvcmNoCiAgICB3b3JrZXJfc2VlZCA9IHRvcmNoLmluaXRpYWxfc2VlZCgpICUgMioqMzIKICAgIG5wLnJhbmRvbS5zZWVkKHdvcmtlcl9zZWVkKQogICAgcmFuZG9tLnNlZWQod29ya2VyX3NlZWQpCiAgICB0b3JjaC5zZXRfbnVtX3RocmVhZHMoMSkKCgpkZWYgbWFrZV9sb2FkZXIoZGY6IHBkLkRhdGFGcmFtZSwgaW1hZ2VzX2Rpcjogc3RyIHwgUGF0aCwgdHJhbnNmb3JtLCBiYXRjaF9zaXplOiBpbnQsCiAgICAgICAgICAgICAgICB0cmFpbjogYm9vbCwgc2FtcGxlcjogc3RyIHwgTm9uZSA9IE5vbmUsIG51bV93b3JrZXJzOiBpbnQgPSAyLAogICAgICAgICAgICAgICAgc2VlZDogaW50ID0gMCk6CiAgICAiIiJDcmVhdGUgYSByZXByb2R1Y2libGUgbG9hZGVyOyBldmFsdWF0aW9uIHByZXNlcnZlcyBDU1Ygcm93IG9yZGVyLiIiIgogICAgaW1wb3J0IG51bXB5IGFzIG5wCiAgICBpbXBvcnQgdG9yY2gKICAgIGZyb20gdG9yY2gudXRpbHMuZGF0YSBpbXBvcnQgRGF0YUxvYWRlciwgV2VpZ2h0ZWRSYW5kb21TYW1wbGVyCgogICAgaWYgYmF0Y2hfc2l6ZSA8IDE6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiYmF0Y2hfc2l6ZSBtdXN0IGJlIHBvc2l0aXZlIikKICAgIGlmIHNhbXBsZXIgbm90IGluIChOb25lLCAiYmFsYW5jZWQiKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiVW5rbm93biBzYW1wbGVyOiB7c2FtcGxlcn0iKQogICAgZ2VuZXJhdG9yID0gdG9yY2guR2VuZXJhdG9yKCkubWFudWFsX3NlZWQoaW50KHNlZWQpKQogICAgZGF0YXNldCA9IERlZXBXZWVkc0RhdGFzZXQoZGYsIGltYWdlc19kaXIsIHRyYW5zZm9ybSkKICAgIHdlaWdodGVkX3NhbXBsZXIgPSBOb25lCiAgICBpZiB0cmFpbiBhbmQgc2FtcGxlciA9PSAiYmFsYW5jZWQiOgogICAgICAgIGxhYmVscyA9IG5wLmFzYXJyYXkoZGF0YXNldC5sYWJlbHMsIGR0eXBlPW5wLmludDY0KQogICAgICAgIGNvdW50cyA9IG5wLmJpbmNvdW50KGxhYmVscywgbWlubGVuZ3RoPU5VTV9DTEFTU0VTKQogICAgICAgIHdlaWdodHMgPSBucC5hc2FycmF5KFsxLjAgLyBjb3VudHNbbGFiZWxdIGZvciBsYWJlbCBpbiBsYWJlbHNdLCBkdHlwZT1ucC5mbG9hdDY0KQogICAgICAgIHdlaWdodGVkX3NhbXBsZXIgPSBXZWlnaHRlZFJhbmRvbVNhbXBsZXIoCiAgICAgICAgICAgIHRvcmNoLmFzX3RlbnNvcih3ZWlnaHRzLCBkdHlwZT10b3JjaC5kb3VibGUpLCBsZW4od2VpZ2h0cyksIHJlcGxhY2VtZW50PVRydWUsCiAgICAgICAgICAgIGdlbmVyYXRvcj1nZW5lcmF0b3IsCiAgICAgICAgKQoKICAgICMgQXQgbW9zdCBvbmUgcXVldWVkIGJhdGNoIHBlciB3b3JrZXI7IG5vIGltYWdlIGNhY2hlIGFuZCBubyBwZXJzaXN0ZW50IHdvcmtlciBwb29sLgogICAgd29ya2VyX29wdGlvbnMgPSB7InByZWZldGNoX2ZhY3RvciI6IDEsICJtdWx0aXByb2Nlc3NpbmdfY29udGV4dCI6ICJzcGF3biJ9IGlmIG51bV93b3JrZXJzID4gMCBlbHNlIHt9CiAgICByZXR1cm4gRGF0YUxvYWRlcigKICAgICAgICBkYXRhc2V0LCBiYXRjaF9zaXplPWJhdGNoX3NpemUsIHNodWZmbGU9Ym9vbCh0cmFpbiBhbmQgd2VpZ2h0ZWRfc2FtcGxlciBpcyBOb25lKSwKICAgICAgICBzYW1wbGVyPXdlaWdodGVkX3NhbXBsZXIsIG51bV93b3JrZXJzPW1heCgwLCBpbnQobnVtX3dvcmtlcnMpKSwKICAgICAgICBwaW5fbWVtb3J5PWJvb2wodG9yY2guY3VkYS5pc19hdmFpbGFibGUoKSksIGRyb3BfbGFzdD1GYWxzZSwKICAgICAgICB3b3JrZXJfaW5pdF9mbj1zZWVkX3dvcmtlciwgZ2VuZXJhdG9yPWdlbmVyYXRvciwKICAgICAgICBwZXJzaXN0ZW50X3dvcmtlcnM9RmFsc2UsICoqd29ya2VyX29wdGlvbnMsCiAgICApCg==",
  "model.py": "IiIiTW9kZWwgY3JlYXRpb24gYW5kIHBhcmFtZXRlciBhY2NvdW50aW5nIGZvciB0aGUgRGVlcFdlZWRzIGV4cGVyaW1lbnRzLiIiIgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgpTVUdHRVNURURfQkFDS0JPTkVTID0gewogICAgInJlc25ldDUwIjogInJlc25ldDUwIiwKICAgICJyZXNuZXh0NTAiOiAicmVzbmV4dDUwXzMyeDRkIiwKICAgICJjb252bmV4dF90aW55IjogImNvbnZuZXh0X3RpbnkiLAogICAgImRlaXRfc21hbGwiOiAiZGVpdF9zbWFsbF9wYXRjaDE2XzIyNCIsCiAgICAic3dpbl90aW55IjogInN3aW5fdGlueV9wYXRjaDRfd2luZG93N18yMjQiLAogICAgImVmZmljaWVudG5ldF9iMCI6ICJlZmZpY2llbnRuZXRfYjAiLAogICAgIm1vYmlsZW5ldHYzIjogIm1vYmlsZW5ldHYzX2xhcmdlXzEwMCIsCn0KCgpkZWYgYnVpbGRfbW9kZWwobmFtZTogc3RyLCBwcmV0cmFpbmVkOiBib29sID0gVHJ1ZSwgbnVtX2NsYXNzZXM6IGludCA9IDksCiAgICAgICAgICAgICAgICBkcm9wX3JhdGU6IGZsb2F0ID0gMC4wLCBpbml0OiBzdHIgPSAiZmluZXR1bmUiKToKICAgICIiIkNyZWF0ZSBhIHRpbW0gY2xhc3NpZmllciBhbmQgcmVjb3JkIHRoZSBwcmV0cmFpbmVkIHRhZyBmb3IgcmVwcm9kdWNpYmlsaXR5LiIiIgogICAgaW1wb3J0IHRpbW0KCiAgICBpZiBpbml0IG5vdCBpbiB7InNjcmF0Y2giLCAiZnJvemVuIiwgImZpbmV0dW5lIn06CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiaW5pdCBtdXN0IGJlIHNjcmF0Y2gsIGZyb3plbiwgb3IgZmluZXR1bmUiKQogICAgbW9kZWxfbmFtZSA9IFNVR0dFU1RFRF9CQUNLQk9ORVMuZ2V0KG5hbWUsIG5hbWUpCiAgICB1c2VfcHJldHJhaW5lZCA9IGJvb2wocHJldHJhaW5lZCBhbmQgaW5pdCAhPSAic2NyYXRjaCIpCiAgICBtb2RlbCA9IHRpbW0uY3JlYXRlX21vZGVsKG1vZGVsX25hbWUsIHByZXRyYWluZWQ9dXNlX3ByZXRyYWluZWQsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIG51bV9jbGFzc2VzPWludChudW1fY2xhc3NlcyksIGRyb3BfcmF0ZT1mbG9hdChkcm9wX3JhdGUpKQogICAgY2ZnID0gZ2V0YXR0cihtb2RlbCwgInByZXRyYWluZWRfY2ZnIiwge30pIG9yIHt9CiAgICBpZiBub3QgaXNpbnN0YW5jZShjZmcsIGRpY3QpOgogICAgICAgIGNmZyA9IGNmZy50b19kaWN0KCkgaWYgaGFzYXR0cihjZmcsICJ0b19kaWN0IikgZWxzZSB2YXJzKGNmZykKICAgIG1vZGVsLnByZXRyYWluZWRfdGFnID0gKChjZmcuZ2V0KCJ0YWciKSBvciBjZmcuZ2V0KCJoZl9odWJfaWQiKSBvciBjZmcuZ2V0KCJ1cmwiKSBvciAicHJldHJhaW5lZCIpCiAgICAgICAgICAgICAgICAgICAgICAgICAgICBpZiB1c2VfcHJldHJhaW5lZCBlbHNlICJzY3JhdGNoIikKICAgIG1vZGVsLm1vZGVsX25hbWUgPSBtb2RlbF9uYW1lCiAgICBpZiBpbml0ID09ICJmcm96ZW4iOgogICAgICAgIGZyZWV6ZV9iYWNrYm9uZShtb2RlbCkKICAgIHJldHVybiBtb2RlbAoKCmRlZiBfY2xhc3NpZmllcl9wYXJhbWV0ZXJzKG1vZGVsKSAtPiBzZXRbaW50XToKICAgIGNsYXNzaWZpZXIgPSBtb2RlbC5nZXRfY2xhc3NpZmllcigpIGlmIGhhc2F0dHIobW9kZWwsICJnZXRfY2xhc3NpZmllciIpIGVsc2UgTm9uZQogICAgaWYgY2xhc3NpZmllciBpcyBOb25lOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIlRoaXMgbW9kZWwgZG9lcyBub3QgZXhwb3NlIGEgY2xhc3NpZmllciB0aHJvdWdoIGdldF9jbGFzc2lmaWVyKCkiKQogICAgcmV0dXJuIHtpZChwYXJhbSkgZm9yIHBhcmFtIGluIGNsYXNzaWZpZXIucGFyYW1ldGVycygpfQoKCmRlZiBmcmVlemVfYmFja2JvbmUobW9kZWwpIC0+IE5vbmU6CiAgICAiIiJGcmVlemUgZmVhdHVyZSBwYXJhbWV0ZXJzIHdoaWxlIGtlZXBpbmcgdGhlIGZpbmFsIGNsYXNzaWZpZXIgdHJhaW5hYmxlLiIiIgogICAgY2xhc3NpZmllcl9pZHMgPSBfY2xhc3NpZmllcl9wYXJhbWV0ZXJzKG1vZGVsKQogICAgaWYgbm90IGNsYXNzaWZpZXJfaWRzOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIlRoZSBjbGFzc2lmaWVyIGhhcyBubyBwYXJhbWV0ZXJzIHRvIHRyYWluIikKICAgIGZvciBwYXJhbSBpbiBtb2RlbC5wYXJhbWV0ZXJzKCk6CiAgICAgICAgcGFyYW0ucmVxdWlyZXNfZ3JhZCA9IGlkKHBhcmFtKSBpbiBjbGFzc2lmaWVyX2lkcwogICAgbW9kZWwuX2ZyZWV6ZV9iYWNrYm9uZSA9IFRydWUKICAgICMgSW1tZWRpYXRlbHkgZnJlZXplIEJhdGNoTm9ybSBzdGF0aXN0aWNzOyB0cmFpbi5ydW4gcmVhcHBsaWVzIHRoaXMgYWZ0ZXIgbW9kZWwudHJhaW4oKS4KICAgIGZvciBtb2R1bGUgaW4gbW9kZWwubW9kdWxlcygpOgogICAgICAgIGlmIG1vZHVsZS5fX2NsYXNzX18uX19uYW1lX18gaW4geyJCYXRjaE5vcm0xZCIsICJCYXRjaE5vcm0yZCIsICJCYXRjaE5vcm0zZCIsICJTeW5jQmF0Y2hOb3JtIn06CiAgICAgICAgICAgIG1vZHVsZS5ldmFsKCkKCgpkZWYgcGFyYW1fZ3JvdXBzKG1vZGVsLCBscl9iYWNrYm9uZTogZmxvYXQsIGxyX2hlYWQ6IGZsb2F0LCB3ZWlnaHRfZGVjYXk6IGZsb2F0KToKICAgICIiIlJldHVybiBzZXBhcmF0ZSBkZWNheS9uby1kZWNheSBncm91cHMgYXQgYmFja2JvbmUgYW5kIGNsYXNzaWZpZXIgbGVhcm5pbmcgcmF0ZXMuIiIiCiAgICBjbGFzc2lmaWVyX2lkcyA9IF9jbGFzc2lmaWVyX3BhcmFtZXRlcnMobW9kZWwpCiAgICBncm91cHMgPSB7ImJhY2tib25lX2RlY2F5IjogW10sICJiYWNrYm9uZV9ub19kZWNheSI6IFtdLAogICAgICAgICAgICAgICJoZWFkX2RlY2F5IjogW10sICJoZWFkX25vX2RlY2F5IjogW119CiAgICBmb3IgbmFtZSwgcGFyYW0gaW4gbW9kZWwubmFtZWRfcGFyYW1ldGVycygpOgogICAgICAgIGlmIG5vdCBwYXJhbS5yZXF1aXJlc19ncmFkOgogICAgICAgICAgICBjb250aW51ZQogICAgICAgIG5vX2RlY2F5ID0gcGFyYW0ubmRpbSA8PSAxIG9yIG5hbWUuZW5kc3dpdGgoIi5iaWFzIikKICAgICAgICBpZiBpZChwYXJhbSkgaW4gY2xhc3NpZmllcl9pZHM6CiAgICAgICAgICAgIGdyb3Vwc1siaGVhZF9ub19kZWNheSIgaWYgbm9fZGVjYXkgZWxzZSAiaGVhZF9kZWNheSJdLmFwcGVuZChwYXJhbSkKICAgICAgICBlbGlmIG5vX2RlY2F5OgogICAgICAgICAgICBncm91cHNbImJhY2tib25lX25vX2RlY2F5Il0uYXBwZW5kKHBhcmFtKQogICAgICAgIGVsc2U6CiAgICAgICAgICAgIGdyb3Vwc1siYmFja2JvbmVfZGVjYXkiXS5hcHBlbmQocGFyYW0pCiAgICByZXN1bHQgPSBbXQogICAgZm9yIGtleSBpbiAoImJhY2tib25lX2RlY2F5IiwgImJhY2tib25lX25vX2RlY2F5IiwgImhlYWRfZGVjYXkiLCAiaGVhZF9ub19kZWNheSIpOgogICAgICAgIHBhcmFtcyA9IGdyb3Vwc1trZXldCiAgICAgICAgaWYgbm90IHBhcmFtczoKICAgICAgICAgICAgY29udGludWUKICAgICAgICBpc19oZWFkID0ga2V5LnN0YXJ0c3dpdGgoImhlYWRfIikKICAgICAgICBub19kZWNheSA9IGtleS5lbmRzd2l0aCgibm9fZGVjYXkiKQogICAgICAgIHJlc3VsdC5hcHBlbmQoeyJwYXJhbXMiOiBwYXJhbXMsCiAgICAgICAgICAgICAgICAgICAgICAgImxyIjogZmxvYXQobHJfaGVhZCBpZiBpc19oZWFkIGVsc2UgbHJfYmFja2JvbmUpLAogICAgICAgICAgICAgICAgICAgICAgICJ3ZWlnaHRfZGVjYXkiOiAwLjAgaWYgbm9fZGVjYXkgZWxzZSBmbG9hdCh3ZWlnaHRfZGVjYXkpfSkKICAgIGlmIG5vdCByZXN1bHQ6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiTW9kZWwgaGFzIG5vIHRyYWluYWJsZSBwYXJhbWV0ZXJzIikKICAgIHJldHVybiByZXN1bHQKCgpkZWYgY291bnRfcGFyYW1zKG1vZGVsKSAtPiBmbG9hdDoKICAgICIiIlJldHVybiBhbGwgcGFyYW1ldGVycywgaW5jbHVkaW5nIGZyb3plbiBvbmVzLCBpbiBtaWxsaW9ucy4iIiIKICAgIHJldHVybiBzdW0ocGFyYW0ubnVtZWwoKSBmb3IgcGFyYW0gaW4gbW9kZWwucGFyYW1ldGVycygpKSAvIDFfMDAwXzAwMC4wCgoKZGVmIGNvdW50X2dtYWNzKG1vZGVsLCBpbWdfc2l6ZTogaW50ID0gMjI0KSAtPiBmbG9hdDoKICAgICIiIkVzdGltYXRlIE1BQ3MgZnJvbSBDb252MmQvTGluZWFyIGxheWVycyB3aXRoIHRlbXBvcmFyeSBmb3J3YXJkIGhvb2tzLgoKICAgIEluY2x1ZGVzIFFLL0FWIG1hdHJpeCBtdWx0aXBsaWNhdGlvbnMgZm9yIHRpbW0gdG9rZW4tYXR0ZW50aW9uIG1vZHVsZXMuIE5vcm1zLAogICAgYWN0aXZhdGlvbnMgYW5kIHBvb2xpbmcgYXJlIGV4Y2x1ZGVkOyB0aGlzIGlzIGFuIGV4cGxpY2l0bHkgbGFiZWxsZWQgTUFDIGVzdGltYXRlLgogICAgIiIiCiAgICBpbXBvcnQgdG9yY2gKICAgIGZyb20gdG9yY2ggaW1wb3J0IG5uCgogICAgaWYgaW1nX3NpemUgPCAxOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoImltZ19zaXplIG11c3QgYmUgcG9zaXRpdmUiKQogICAgdG90YWwgPSBbMF0KICAgIGhvb2tzID0gW10KCiAgICBkZWYgY29udl9tYWNzKGxheWVyLCBfaW5wdXRzLCBvdXRwdXQpOgogICAgICAgIGtoLCBrdyA9IGxheWVyLmtlcm5lbF9zaXplCiAgICAgICAgdG90YWxbMF0gKz0gb3V0cHV0Lm51bWVsKCkgKiAobGF5ZXIuaW5fY2hhbm5lbHMgLy8gbGF5ZXIuZ3JvdXBzKSAqIGtoICoga3cKCiAgICBkZWYgbGluZWFyX21hY3MobGF5ZXIsIF9pbnB1dHMsIG91dHB1dCk6CiAgICAgICAgdG90YWxbMF0gKz0gb3V0cHV0Lm51bWVsKCkgKiBsYXllci5pbl9mZWF0dXJlcwoKICAgIGRlZiBhdHRlbnRpb25fbWFjcyhsYXllciwgaW5wdXRzLCBvdXRwdXQpOgogICAgICAgIHggPSBpbnB1dHNbMF0KICAgICAgICBpZiB4Lm5kaW0gPT0gMzoKICAgICAgICAgICAgYiwgbiwgYyA9IHguc2hhcGUKICAgICAgICAgICAgdG90YWxbMF0gKz0gMiAqIGIgKiBuICogbiAqIGMKCiAgICBmb3IgbGF5ZXIgaW4gbW9kZWwubW9kdWxlcygpOgogICAgICAgIGlmIGlzaW5zdGFuY2UobGF5ZXIsIG5uLkNvbnYyZCk6CiAgICAgICAgICAgIGhvb2tzLmFwcGVuZChsYXllci5yZWdpc3Rlcl9mb3J3YXJkX2hvb2soY29udl9tYWNzKSkKICAgICAgICBlbGlmIGlzaW5zdGFuY2UobGF5ZXIsIG5uLkxpbmVhcik6CiAgICAgICAgICAgIGhvb2tzLmFwcGVuZChsYXllci5yZWdpc3Rlcl9mb3J3YXJkX2hvb2sobGluZWFyX21hY3MpKQogICAgICAgIGlmIGhhc2F0dHIobGF5ZXIsICJxa3YiKSBhbmQgaGFzYXR0cihsYXllciwgIm51bV9oZWFkcyIpOgogICAgICAgICAgICBob29rcy5hcHBlbmQobGF5ZXIucmVnaXN0ZXJfZm9yd2FyZF9ob29rKGF0dGVudGlvbl9tYWNzKSkKICAgIHRyYWluaW5nX3N0YXRlcyA9IFsobW9kdWxlLCBtb2R1bGUudHJhaW5pbmcpIGZvciBtb2R1bGUgaW4gbW9kZWwubW9kdWxlcygpXQogICAgdHJ5OgogICAgICAgIHBhcmFtZXRlciA9IG5leHQobW9kZWwucGFyYW1ldGVycygpKQogICAgICAgIGRldmljZSA9IHBhcmFtZXRlci5kZXZpY2UKICAgICAgICBkdHlwZSA9IHBhcmFtZXRlci5kdHlwZSBpZiBwYXJhbWV0ZXIuaXNfZmxvYXRpbmdfcG9pbnQoKSBlbHNlIHRvcmNoLmZsb2F0MzIKICAgICAgICBtb2RlbC5ldmFsKCkKICAgICAgICB3aXRoIHRvcmNoLmluZmVyZW5jZV9tb2RlKCk6CiAgICAgICAgICAgIG1vZGVsKHRvcmNoLnplcm9zKDEsIDMsIGltZ19zaXplLCBpbWdfc2l6ZSwgZGV2aWNlPWRldmljZSwgZHR5cGU9ZHR5cGUpKQogICAgZmluYWxseToKICAgICAgICBmb3IgaG9vayBpbiBob29rczoKICAgICAgICAgICAgaG9vay5yZW1vdmUoKQogICAgICAgIGZvciBtb2R1bGUsIHdhc190cmFpbmluZyBpbiB0cmFpbmluZ19zdGF0ZXM6CiAgICAgICAgICAgIG1vZHVsZS50cmFpbmluZyA9IHdhc190cmFpbmluZwogICAgbW9kZWwuZ21hY19tZXRob2QgPSAiQ29udjJkICsgTGluZWFyICsgYXR0ZW50aW9uIFFLL0FWIE1BQ3M7IGV4Y2x1ZGVzIG5vcm0vYWN0aXZhdGlvbi9wb29saW5nIgogICAgcmV0dXJuIHRvdGFsWzBdIC8gMV8wMDBfMDAwXzAwMC4wCg==",
  "losses.py": "IiIiQ2xhc3NpZmljYXRpb24gbG9zc2VzIGFuZCBiYXRjaC1sZXZlbCBNaXh1cC9DdXRNaXggaGVscGVycy4iIiIKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IHRvcmNoCmZyb20gdG9yY2ggaW1wb3J0IG5uCmZyb20gdG9yY2gubm4gaW1wb3J0IGZ1bmN0aW9uYWwgYXMgRgoKCmRlZiBidWlsZF9jcml0ZXJpb24oa2luZDogc3RyID0gImNlIiwgKiprdyk6CiAgICBraW5kID0ga2luZC5sb3dlcigpCiAgICB3ZWlnaHQgPSBrdy5nZXQoIndlaWdodCIpCiAgICBpZiBraW5kID09ICJjZSI6CiAgICAgICAgcmV0dXJuIG5uLkNyb3NzRW50cm9weUxvc3Mod2VpZ2h0PXdlaWdodCkKICAgIGlmIGtpbmQgPT0gImxzIjoKICAgICAgICByZXR1cm4gTGFiZWxTbW9vdGhpbmdDRShrdy5nZXQoInNtb290aGluZyIsIDAuMSksIHdlaWdodD13ZWlnaHQpCiAgICBpZiBraW5kID09ICJmb2NhbCI6CiAgICAgICAgcmV0dXJuIEZvY2FsTG9zcyhrdy5nZXQoImdhbW1hIiwgMi4wKSwgYWxwaGE9a3cuZ2V0KCJhbHBoYSIsIHdlaWdodCkpCiAgICBpZiBraW5kID09ICJjZV93ZWlnaHRlZCI6CiAgICAgICAgaWYgd2VpZ2h0IGlzIE5vbmU6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoImNlX3dlaWdodGVkIHJlcXVpcmVzIHdlaWdodD1jbGFzc193ZWlnaHRzKC4uLikiKQogICAgICAgIHJldHVybiBubi5Dcm9zc0VudHJvcHlMb3NzKHdlaWdodD13ZWlnaHQpCiAgICByYWlzZSBWYWx1ZUVycm9yKGYiVW5rbm93biBsb3NzIGtpbmQ6IHtraW5kfSIpCgoKY2xhc3MgTGFiZWxTbW9vdGhpbmdDRShubi5Nb2R1bGUpOgogICAgIiIiQ3Jvc3MgZW50cm9weSB3aXRoIHVuaWZvcm0gbGFiZWwgc21vb3RoaW5nIGFuZCBvcHRpb25hbCBjbGFzcyB3ZWlnaHRzLiIiIgoKICAgIGRlZiBfX2luaXRfXyhzZWxmLCBzbW9vdGhpbmc6IGZsb2F0ID0gMC4xLCB3ZWlnaHQ9Tm9uZSk6CiAgICAgICAgc3VwZXIoKS5fX2luaXRfXygpCiAgICAgICAgaWYgbm90IDAuMCA8PSBmbG9hdChzbW9vdGhpbmcpIDwgMS4wOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJzbW9vdGhpbmcgbXVzdCBiZSBpbiBbMCwgMSkiKQogICAgICAgIHNlbGYuc21vb3RoaW5nID0gZmxvYXQoc21vb3RoaW5nKQogICAgICAgIHNlbGYucmVnaXN0ZXJfYnVmZmVyKCJ3ZWlnaHQiLCBOb25lIGlmIHdlaWdodCBpcyBOb25lIGVsc2UgdG9yY2guYXNfdGVuc29yKHdlaWdodCkuZGV0YWNoKCkuY2xvbmUoKSkKCiAgICBkZWYgZm9yd2FyZChzZWxmLCBsb2dpdHMsIHRhcmdldCk6CiAgICAgICAgcmV0dXJuIEYuY3Jvc3NfZW50cm9weShsb2dpdHMsIHRhcmdldCwgd2VpZ2h0PXNlbGYud2VpZ2h0LAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgbGFiZWxfc21vb3RoaW5nPXNlbGYuc21vb3RoaW5nKQoKCmNsYXNzIEZvY2FsTG9zcyhubi5Nb2R1bGUpOgogICAgIiIiTXVsdGljbGFzcyBmb2NhbCBsb3NzOyBnYW1tYT0wIHdpdGggbm8gYWxwaGEgaXMgc3RhbmRhcmQgY3Jvc3MgZW50cm9weS4iIiIKCiAgICBkZWYgX19pbml0X18oc2VsZiwgZ2FtbWE6IGZsb2F0ID0gMi4wLCBhbHBoYT1Ob25lKToKICAgICAgICBzdXBlcigpLl9faW5pdF9fKCkKICAgICAgICBpZiBmbG9hdChnYW1tYSkgPCAwOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJnYW1tYSBtdXN0IGJlIG5vbm5lZ2F0aXZlIikKICAgICAgICBzZWxmLmdhbW1hID0gZmxvYXQoZ2FtbWEpCiAgICAgICAgc2VsZi5yZWdpc3Rlcl9idWZmZXIoImFscGhhIiwgTm9uZSBpZiBhbHBoYSBpcyBOb25lIGVsc2UgdG9yY2guYXNfdGVuc29yKGFscGhhLCBkdHlwZT10b3JjaC5mbG9hdDMyKSkKCiAgICBkZWYgZm9yd2FyZChzZWxmLCBsb2dpdHMsIHRhcmdldCk6CiAgICAgICAgbG9nX3Byb2JzID0gRi5sb2dfc29mdG1heChsb2dpdHMsIGRpbT0xKQogICAgICAgIHRhcmdldCA9IHRhcmdldC5sb25nKCkKICAgICAgICBsb2dfcHQgPSBsb2dfcHJvYnMuZ2F0aGVyKDEsIHRhcmdldC52aWV3KC0xLCAxKSkuc3F1ZWV6ZSgxKQogICAgICAgIGxvc3MgPSAtKCgxLjAgLSBsb2dfcHQuZXhwKCkpLmNsYW1wX21pbigwLjApICoqIHNlbGYuZ2FtbWEpICogbG9nX3B0CiAgICAgICAgaWYgc2VsZi5hbHBoYSBpcyBub3QgTm9uZToKICAgICAgICAgICAgbG9zcyA9IGxvc3MgKiBzZWxmLmFscGhhLnRvKGRldmljZT1sb2dpdHMuZGV2aWNlLCBkdHlwZT1sb2dpdHMuZHR5cGUpW3RhcmdldF0KICAgICAgICByZXR1cm4gbG9zcy5tZWFuKCkKCgpkZWYgY2xhc3Nfd2VpZ2h0cyhjb3VudHMsIGJldGE6IGZsb2F0ID0gMC4wKToKICAgIGNvdW50cyA9IHRvcmNoLmFzX3RlbnNvcihjb3VudHMsIGR0eXBlPXRvcmNoLmZsb2F0NjQpCiAgICBpZiBjb3VudHMubmRpbSAhPSAxIG9yIGNvdW50cy5udW1lbCgpICE9IDkgb3IgdG9yY2guYW55KGNvdW50cyA8PSAwKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJjb3VudHMgbXVzdCBjb250YWluIDkgcG9zaXRpdmUgdHJhaW5pbmctc2V0IGNsYXNzIGNvdW50cyIpCiAgICBiZXRhID0gZmxvYXQoYmV0YSkKICAgIGlmIGJldGEgPCAwIG9yIGJldGEgPj0gMToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJiZXRhIG11c3QgYmUgaW4gWzAsIDEpIikKICAgIHdlaWdodHMgPSBjb3VudHMucmVjaXByb2NhbCgpIGlmIGJldGEgPT0gMCBlbHNlICgxLjAgLSBiZXRhKSAvICgxLjAgLSB0b3JjaC5wb3coYmV0YSwgY291bnRzKSkKICAgIHJldHVybiAod2VpZ2h0cyAvIHdlaWdodHMubWVhbigpKS50byhkdHlwZT10b3JjaC5mbG9hdDMyKQoKCmRlZiBtaXhfYmF0Y2goeCwgeSwgYWxwaGE6IGZsb2F0ID0gMS4wLCBtb2RlOiBzdHIgPSAiY3V0bWl4Iik6CiAgICBpZiB4Lm5kaW0gIT0gNCBvciBsZW4oeSkgIT0gbGVuKHgpOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoInggbXVzdCBiZSBOQ0hXIGFuZCB5IG11c3QgaGF2ZSB0aGUgc2FtZSBiYXRjaCBsZW5ndGgiKQogICAgaWYgYWxwaGEgPD0gMDoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJhbHBoYSBtdXN0IGJlIHBvc2l0aXZlIikKICAgIGlmIG1vZGUgbm90IGluIHsibWl4dXAiLCAiY3V0bWl4In06CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigibW9kZSBtdXN0IGJlIG1peHVwIG9yIGN1dG1peCIpCiAgICBpZiBsZW4oeCkgPCAyOgogICAgICAgIHJldHVybiB4LCAoeSwgeSwgMS4wKQogICAgbGFtID0gZmxvYXQodG9yY2guZGlzdHJpYnV0aW9ucy5CZXRhKGFscGhhLCBhbHBoYSkuc2FtcGxlKCkuaXRlbSgpKQogICAgcGVybXV0YXRpb24gPSB0b3JjaC5yYW5kcGVybSh4LnNpemUoMCksIGRldmljZT14LmRldmljZSkKICAgIHlfYSwgeV9iID0geSwgeVtwZXJtdXRhdGlvbl0KICAgIGlmIG1vZGUgPT0gIm1peHVwIjoKICAgICAgICByZXR1cm4gbGFtICogeCArICgxLjAgLSBsYW0pICogeFtwZXJtdXRhdGlvbl0sICh5X2EsIHlfYiwgbGFtKQoKICAgIGhlaWdodCwgd2lkdGggPSB4LnNoYXBlWy0yOl0KICAgIGN1dF9yYXRpbyA9ICgxLjAgLSBsYW0pICoqIDAuNQogICAgY3V0X2gsIGN1dF93ID0gaW50KGhlaWdodCAqIGN1dF9yYXRpbyksIGludCh3aWR0aCAqIGN1dF9yYXRpbykKICAgIGNlbnRlcl95ID0gaW50KHRvcmNoLnJhbmRpbnQoaGVpZ2h0LCAoMSwpLCBkZXZpY2U9eC5kZXZpY2UpLml0ZW0oKSkKICAgIGNlbnRlcl94ID0gaW50KHRvcmNoLnJhbmRpbnQod2lkdGgsICgxLCksIGRldmljZT14LmRldmljZSkuaXRlbSgpKQogICAgeTEsIHkyID0gbWF4KGNlbnRlcl95IC0gY3V0X2ggLy8gMiwgMCksIG1pbihjZW50ZXJfeSArIChjdXRfaCArIDEpIC8vIDIsIGhlaWdodCkKICAgIHgxLCB4MiA9IG1heChjZW50ZXJfeCAtIGN1dF93IC8vIDIsIDApLCBtaW4oY2VudGVyX3ggKyAoY3V0X3cgKyAxKSAvLyAyLCB3aWR0aCkKICAgIG1peGVkID0geC5jbG9uZSgpCiAgICBtaXhlZFs6LCA6LCB5MTp5MiwgeDE6eDJdID0geFtwZXJtdXRhdGlvbiwgOiwgeTE6eTIsIHgxOngyXQogICAgbGFtID0gMS4wIC0gKCh5MiAtIHkxKSAqICh4MiAtIHgxKSAvIGZsb2F0KGhlaWdodCAqIHdpZHRoKSkKICAgIHJldHVybiBtaXhlZCwgKHlfYSwgeV9iLCBsYW0pCgoKZGVmIG1peGVkX2xvc3MoY3JpdGVyaW9uLCBsb2dpdHMsIHRhcmdldHMpOgogICAgeV9hLCB5X2IsIGxhbSA9IHRhcmdldHMKICAgIGxhbSA9IGZsb2F0KGxhbSkKICAgIHJldHVybiBsYW0gKiBjcml0ZXJpb24obG9naXRzLCB5X2EpICsgKDEuMCAtIGxhbSkgKiBjcml0ZXJpb24obG9naXRzLCB5X2IpCg==",
  "benchmark.py": "IiIiRm9yd2FyZC1wYXNzIGxhdGVuY3kgbWVhc3VyZW1lbnQgd2l0aCB3YXJtdXAgYW5kIGV4cGxpY2l0IEdQVSBzeW5jaHJvbml6YXRpb24uIiIiCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCB0aW1lCgoKZGVmIGJlbmNoKGZuLCB3YXJtdXA6IGludCA9IDEwLCBpdGVyczogaW50ID0gMTAwLCBzeW5jPU5vbmUpIC0+IGRpY3Q6CiAgICBpbXBvcnQgbnVtcHkgYXMgbnAKCiAgICB3YXJtdXAsIGl0ZXJzID0gbWF4KDEwLCBpbnQod2FybXVwKSksIG1heCg1MCwgaW50KGl0ZXJzKSkKICAgIGZvciBfIGluIHJhbmdlKHdhcm11cCk6CiAgICAgICAgZm4oKQogICAgaWYgc3luYzoKICAgICAgICBzeW5jKCkKICAgIHNhbXBsZXMgPSBbXQogICAgZm9yIF8gaW4gcmFuZ2UoaXRlcnMpOgogICAgICAgIGlmIHN5bmM6CiAgICAgICAgICAgIHN5bmMoKQogICAgICAgIHN0YXJ0ID0gdGltZS5wZXJmX2NvdW50ZXIoKQogICAgICAgIGZuKCkKICAgICAgICBpZiBzeW5jOgogICAgICAgICAgICBzeW5jKCkKICAgICAgICBzYW1wbGVzLmFwcGVuZCgodGltZS5wZXJmX2NvdW50ZXIoKSAtIHN0YXJ0KSAqIDFfMDAwLjApCiAgICByZXR1cm4geyJwNTAiOiBmbG9hdChucC5wZXJjZW50aWxlKHNhbXBsZXMsIDUwKSksCiAgICAgICAgICAgICJwOTUiOiBmbG9hdChucC5wZXJjZW50aWxlKHNhbXBsZXMsIDk1KSksCiAgICAgICAgICAgICJwOTkiOiBmbG9hdChucC5wZXJjZW50aWxlKHNhbXBsZXMsIDk5KSksCiAgICAgICAgICAgICJtZWFuIjogZmxvYXQobnAubWVhbihzYW1wbGVzKSksICJuIjogbGVuKHNhbXBsZXMpLAogICAgICAgICAgICAid2FybXVwIjogd2FybXVwfQoKCmRlZiBfcHJlcGFyZV9tb2RlbF9pbnB1dChtb2RlbCwgYmF0Y2hfc2l6ZSwgaW1nX3NpemUsIGR0eXBlLCBkZXZpY2UpOgogICAgaW1wb3J0IGNvcHkKICAgIGltcG9ydCB0b3JjaAoKICAgIGlmIGJhdGNoX3NpemUgPCAxIG9yIGltZ19zaXplIDwgMToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJiYXRjaF9zaXplIGFuZCBpbWdfc2l6ZSBtdXN0IGJlIHBvc2l0aXZlIikKICAgIHRhcmdldCA9IHRvcmNoLmRldmljZShkZXZpY2UpCiAgICBpZiB0YXJnZXQudHlwZSA9PSAiY3VkYSIgYW5kIG5vdCB0b3JjaC5jdWRhLmlzX2F2YWlsYWJsZSgpOgogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigiQ1VEQSByZXF1ZXN0ZWQgYnV0IG5vIEdQVSBpcyBhdmFpbGFibGUiKQogICAgIyBBdm9pZCBhbm90aGVyIGZ1bGwgbW9kZWwgZm9yIHRoZSBub3JtYWwgRlAzMi9BTVAgcGF0aC4KICAgIG1lYXN1cmVkX21vZGVsID0gY29weS5kZWVwY29weShtb2RlbCkudG8odGFyZ2V0KSBpZiBkdHlwZSA9PSAiZnAxNiIgZWxzZSBtb2RlbC50byh0YXJnZXQpCiAgICBpZiBkdHlwZSA9PSAiZnAxNiI6CiAgICAgICAgbWVhc3VyZWRfbW9kZWwgPSBtZWFzdXJlZF9tb2RlbC5oYWxmKCkKICAgIG1lYXN1cmVkX21vZGVsLmV2YWwoKQogICAgaW5wdXRfZHR5cGUgPSB0b3JjaC5mbG9hdDE2IGlmIGR0eXBlID09ICJmcDE2IiBlbHNlIHRvcmNoLmZsb2F0MzIKICAgIHggPSB0b3JjaC5yYW5kbihiYXRjaF9zaXplLCAzLCBpbWdfc2l6ZSwgaW1nX3NpemUsIGRldmljZT10YXJnZXQsIGR0eXBlPWlucHV0X2R0eXBlKQogICAgcmV0dXJuIG1lYXN1cmVkX21vZGVsLCB4LCB0YXJnZXQKCgpkZWYgbGF0ZW5jeV9yZXBvcnQobW9kZWwsIGJhdGNoX3NpemU6IGludCwgaW1nX3NpemU6IGludCwgZHR5cGU6IHN0ciA9ICJmcDMyIiwgZGV2aWNlOiBzdHIgPSAiY3VkYSIsCiAgICAgICAgICAgICAgICAgICB3YXJtdXA6IGludCA9IDEwLCBpdGVyczogaW50ID0gMTAwKSAtPiBkaWN0OgogICAgaW1wb3J0IHRvcmNoCgogICAgZHR5cGUgPSBkdHlwZS5sb3dlcigpCiAgICBpZiBkdHlwZSBub3QgaW4geyJmcDMyIiwgImFtcCIsICJmcDE2In06CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiZHR5cGUgbXVzdCBiZSBmcDMyLCBhbXAsIG9yIGZwMTYiKQogICAgbWVhc3VyZWRfbW9kZWwsIHgsIHRhcmdldCA9IF9wcmVwYXJlX21vZGVsX2lucHV0KG1vZGVsLCBiYXRjaF9zaXplLCBpbWdfc2l6ZSwgZHR5cGUsIGRldmljZSkKICAgIHVzZV9hbXAgPSBkdHlwZSA9PSAiYW1wIiBhbmQgdGFyZ2V0LnR5cGUgPT0gImN1ZGEiCgogICAgZGVmIGZvcndhcmQoKToKICAgICAgICB3aXRoIHRvcmNoLmluZmVyZW5jZV9tb2RlKCk6CiAgICAgICAgICAgIHdpdGggdG9yY2guYXV0b2Nhc3QoZGV2aWNlX3R5cGU9dGFyZ2V0LnR5cGUsIGR0eXBlPXRvcmNoLmZsb2F0MTYsIGVuYWJsZWQ9dXNlX2FtcCk6CiAgICAgICAgICAgICAgICBtZWFzdXJlZF9tb2RlbCh4KQoKICAgIHN5bmMgPSAobGFtYmRhOiB0b3JjaC5jdWRhLnN5bmNocm9uaXplKHRhcmdldCkpIGlmIHRhcmdldC50eXBlID09ICJjdWRhIiBlbHNlIE5vbmUKICAgIHRpbWluZyA9IGJlbmNoKGZvcndhcmQsIHdhcm11cD13YXJtdXAsIGl0ZXJzPWl0ZXJzLCBzeW5jPXN5bmMpCiAgICB0aW1pbmcudXBkYXRlKHsKICAgICAgICAiZ3B1IjogdG9yY2guY3VkYS5nZXRfZGV2aWNlX25hbWUodGFyZ2V0KSBpZiB0YXJnZXQudHlwZSA9PSAiY3VkYSIgZWxzZSAiQ1BVIiwKICAgICAgICAiZHR5cGUiOiBkdHlwZSwgImJhdGNoIjogaW50KGJhdGNoX3NpemUpLCAiaW1nX3NpemUiOiBpbnQoaW1nX3NpemUpLAogICAgICAgICJpbWFnZXNfcGVyX3MiOiBmbG9hdChiYXRjaF9zaXplIC8gKHRpbWluZ1sicDUwIl0gLyAxMDAwLjApKSwKICAgICAgICAidG9yY2giOiB0b3JjaC5fX3ZlcnNpb25fXywgInByZXByb2Nlc3NpbmciOiAiZXhjbHVkZWQiLCAiYm5fZnVzZWQiOiBGYWxzZSwKICAgIH0pCiAgICByZXR1cm4gdGltaW5nCgoKZGVmIHR0YV9sYXRlbmN5KG1vZGVsLCBrX3ZpZXdzOiBpbnQsICoqa3cpIC0+IGRpY3Q6CiAgICAiIiJGb3J3YXJkLW9ubHkgSy12aWV3IG1pY3JvYmVuY2htYXJrOyB1c2UgbWV0aG9kX2xhdGVuY3kgZm9yIGFjdHVhbCBUVEEvYWdncmVnYXRpb24uIiIiCiAgICBpbXBvcnQgdG9yY2gKCiAgICBpZiBpbnQoa192aWV3cykgPCAxOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoImtfdmlld3MgbXVzdCBiZSBwb3NpdGl2ZSIpCiAgICBiYXRjaF9zaXplID0gaW50KGt3LmdldCgiYmF0Y2hfc2l6ZSIsIDEpKQogICAgaW1nX3NpemUgPSBpbnQoa3cuZ2V0KCJpbWdfc2l6ZSIsIDIyNCkpCiAgICBkdHlwZSA9IHN0cihrdy5nZXQoImR0eXBlIiwgImZwMzIiKSkubG93ZXIoKQogICAgaWYgZHR5cGUgbm90IGluIHsiZnAzMiIsICJhbXAiLCAiZnAxNiJ9OgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoImR0eXBlIG11c3QgYmUgZnAzMiwgYW1wLCBvciBmcDE2IikKICAgIGRldmljZSA9IHN0cihrdy5nZXQoImRldmljZSIsICJjdWRhIikpCiAgICB3YXJtdXAsIGl0ZXJzID0gaW50KGt3LmdldCgid2FybXVwIiwgMTApKSwgaW50KGt3LmdldCgiaXRlcnMiLCAxMDApKQogICAgbWVhc3VyZWRfbW9kZWwsIHgsIHRhcmdldCA9IF9wcmVwYXJlX21vZGVsX2lucHV0KG1vZGVsLCBiYXRjaF9zaXplLCBpbWdfc2l6ZSwgZHR5cGUsIGRldmljZSkKICAgIHVzZV9hbXAgPSBkdHlwZSA9PSAiYW1wIiBhbmQgdGFyZ2V0LnR5cGUgPT0gImN1ZGEiCgogICAgZGVmIGZvcndhcmRfdmlld3MoKToKICAgICAgICB3aXRoIHRvcmNoLmluZmVyZW5jZV9tb2RlKCk6CiAgICAgICAgICAgIHdpdGggdG9yY2guYXV0b2Nhc3QoZGV2aWNlX3R5cGU9dGFyZ2V0LnR5cGUsIGR0eXBlPXRvcmNoLmZsb2F0MTYsIGVuYWJsZWQ9dXNlX2FtcCk6CiAgICAgICAgICAgICAgICBmb3IgXyBpbiByYW5nZShpbnQoa192aWV3cykpOgogICAgICAgICAgICAgICAgICAgIG1lYXN1cmVkX21vZGVsKHgpCgogICAgc3luYyA9IChsYW1iZGE6IHRvcmNoLmN1ZGEuc3luY2hyb25pemUodGFyZ2V0KSkgaWYgdGFyZ2V0LnR5cGUgPT0gImN1ZGEiIGVsc2UgTm9uZQogICAgdGltaW5nID0gYmVuY2goZm9yd2FyZF92aWV3cywgd2FybXVwPXdhcm11cCwgaXRlcnM9aXRlcnMsIHN5bmM9c3luYykKICAgIHRpbWluZy51cGRhdGUoeyJncHUiOiB0b3JjaC5jdWRhLmdldF9kZXZpY2VfbmFtZSh0YXJnZXQpIGlmIHRhcmdldC50eXBlID09ICJjdWRhIiBlbHNlICJDUFUiLAogICAgICAgICAgICAgICAgICAgImR0eXBlIjogZHR5cGUsICJiYXRjaCI6IGJhdGNoX3NpemUsICJpbWdfc2l6ZSI6IGltZ19zaXplLAogICAgICAgICAgICAgICAgICAgImltYWdlc19wZXJfcyI6IGZsb2F0KGJhdGNoX3NpemUgLyAodGltaW5nWyJwNTAiXSAvIDEwMDAuMCkpLAogICAgICAgICAgICAgICAgICAgInRvcmNoIjogdG9yY2guX192ZXJzaW9uX18sICJrX3ZpZXdzIjogaW50KGtfdmlld3MpLAogICAgICAgICAgICAgICAgICAgInByZXByb2Nlc3NpbmciOiAiZXhjbHVkZWQiLCAiYm5fZnVzZWQiOiBGYWxzZX0pCiAgICByZXR1cm4gdGltaW5nCgoKZGVmIG1ldGhvZF9sYXRlbmN5KG1vZGVsLCBtZXRob2Q9IkkwMCIsIHNwYWNlPSJwcm9iIiwgdGVtcGVyYXR1cmU9Tm9uZSwgYmF0Y2hfc2l6ZT0xLAogICAgICAgICAgICAgICAgICAgaW1nX3NpemU9MjI0LCBkZXZpY2U9ImN1ZGEiLCB3YXJtdXA9MTAsIGl0ZXJzPTUwKToKICAgICIiIlRpbWUgdGhlIGFjdHVhbCB2aWV3IGNyZWF0aW9uLCBmb3J3YXJkLCBhZ2dyZWdhdGlvbiBhbmQgb3B0aW9uYWwgY2FsaWJyYXRpb24uCgogICAgRXhjbHVkZXMgZmlsZSBkZWNvZGluZywgYmFzZSByZXNpemUvbm9ybWFsaXphdGlvbiBhbmQgaG9zdC10by1kZXZpY2UgdHJhbnNmZXIuCiAgICAiIiIKICAgIGltcG9ydCB0b3JjaAogICAgZnJvbSBpbmZlcmVuY2UgaW1wb3J0IGZvcndhcmRfcHJvYmFiaWxpdGllcwogICAgdGFyZ2V0ID0gdG9yY2guZGV2aWNlKGRldmljZSkKICAgIG1vZGVsLmV2YWwoKQogICAgeCA9IHRvcmNoLnplcm9zKGJhdGNoX3NpemUsIDMsIGltZ19zaXplLCBpbWdfc2l6ZSwgZGV2aWNlPXRhcmdldCkKICAgIGRlZiBmb3J3YXJkKCk6CiAgICAgICAgd2l0aCB0b3JjaC5pbmZlcmVuY2VfbW9kZSgpOgogICAgICAgICAgICByZXR1cm4gZm9yd2FyZF9wcm9iYWJpbGl0aWVzKG1vZGVsLCB4LCBtZXRob2QsIHNwYWNlLCBtYXgoMzIsaW1nX3NpemUtMzIpLCB0ZW1wZXJhdHVyZSkKICAgIHRpbWluZyA9IGJlbmNoKGZvcndhcmQsIHdhcm11cCwgaXRlcnMsCiAgICAgICAgICAgICAgICAgICAobGFtYmRhOiB0b3JjaC5jdWRhLnN5bmNocm9uaXplKHRhcmdldCkpIGlmIHRhcmdldC50eXBlID09ICJjdWRhIiBlbHNlIE5vbmUpCiAgICB0aW1pbmcudXBkYXRlKGdwdT10b3JjaC5jdWRhLmdldF9kZXZpY2VfbmFtZSh0YXJnZXQpIGlmIHRhcmdldC50eXBlID09ICJjdWRhIiBlbHNlICJDUFUiLAogICAgICAgICAgICAgICAgICBkdHlwZT0iYW1wIiBpZiBtZXRob2QgPT0gIkkwOCIgZWxzZSAiZnAzMiIsIGJhdGNoPWJhdGNoX3NpemUsIGltZ19zaXplPWltZ19zaXplLAogICAgICAgICAgICAgICAgICBpbWFnZXNfcGVyX3M9YmF0Y2hfc2l6ZSAvICh0aW1pbmdbInA1MCJdIC8gMTAwMCksIHRvcmNoPXRvcmNoLl9fdmVyc2lvbl9fLAogICAgICAgICAgICAgICAgICBwcmVwcm9jZXNzaW5nPSJiYXNlIHByZXByb2Nlc3NpbmcvSDJEIGV4Y2x1ZGVkOyBUVEEgYW5kIGFnZ3JlZ2F0aW9uIGluY2x1ZGVkIiwgYm5fZnVzZWQ9RmFsc2UpCiAgICByZXR1cm4gdGltaW5nCg==",
  "inference.py": "IiIiSW5mZXJlbmNlIHZpZXdzLCBhZ2dyZWdhdGlvbiwgY2FsaWJyYXRpb24sIGFuZCBDb252L0JhdGNoTm9ybSBmb2xkaW5nLiIiIgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgoKZGVmIGZvcndhcmRfcHJvYmFiaWxpdGllcyhtb2RlbCwgaW1hZ2VzLCBtZXRob2Q9IkkwMCIsIHNwYWNlPSJwcm9iIiwgY3JvcD1Ob25lLCB0ZW1wZXJhdHVyZT1Ob25lLAogICAgICAgICAgICAgICAgICAgICAgICAgIHJldHVybl9sb2dpdHM9RmFsc2UpOgogICAgIiIiU2FtZSB0ZW5zb3IgcGF0aCBmb3IgdmFsaWRhdGlvbiwgZmluYWwgcHJlZGljdGlvbiBhbmQgbGF0ZW5jeSBtZWFzdXJlbWVudC4iIiIKICAgIGltcG9ydCB0b3JjaAogICAgaW1wb3J0IHRvcmNoLm5uLmZ1bmN0aW9uYWwgYXMgRgogICAgaWYgbWV0aG9kIG5vdCBpbiB7IkkwMCIsICJJMDEiLCAiSTAyIiwgIkkwMyIsICJJMDgifToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiVW5rbm93biBpbmZlcmVuY2UgbWV0aG9kIHttZXRob2R9IikKICAgIHdpdGggdG9yY2guYXV0b2Nhc3QoZGV2aWNlX3R5cGU9aW1hZ2VzLmRldmljZS50eXBlLCBkdHlwZT10b3JjaC5mbG9hdDE2LAogICAgICAgICAgICAgICAgICAgICAgICBlbmFibGVkPW1ldGhvZCA9PSAiSTA4IiBhbmQgaW1hZ2VzLmRldmljZS50eXBlID09ICJjdWRhIik6CiAgICAgICAgaWYgbWV0aG9kID09ICJJMDIiOgogICAgICAgICAgICAjIFByb2Nlc3Mgdmlld3Mgc2VxdWVudGlhbGx5OiBvbmx5IHNtYWxsIE4geCA5IG91dHB1dHMgYXJlIHJldGFpbmVkLgogICAgICAgICAgICB2aWV3cyA9IHZpZXdzX211bHRpY3JvcChpbWFnZXMsIGNyb3Agb3IgbWF4KDMyLCBpbWFnZXMuc2hhcGVbLTFdIC0gMzIpKQogICAgICAgICAgICBsb2dpdHMgPSBbbW9kZWwoRi5pbnRlcnBvbGF0ZSh2LCBzaXplPWltYWdlcy5zaGFwZVstMjpdLCBtb2RlPSJiaWxpbmVhciIsIGFsaWduX2Nvcm5lcnM9RmFsc2UpKS5mbG9hdCgpCiAgICAgICAgICAgICAgICAgICAgICBmb3IgdiBpbiB2aWV3c10KICAgICAgICBlbGlmIG1ldGhvZCBpbiB7IkkwMSIsICJJMDMifToKICAgICAgICAgICAgbG9naXRzID0gW21vZGVsKGltYWdlcykuZmxvYXQoKSwgbW9kZWwodmlld19oZmxpcChpbWFnZXMpKS5mbG9hdCgpXQogICAgICAgIGVsc2U6CiAgICAgICAgICAgIGxvZ2l0cyA9IFttb2RlbChpbWFnZXMpLmZsb2F0KCldCiAgICBpZiBzcGFjZSA9PSAicHJvYiI6CiAgICAgICAgcHJvYnMgPSB0b3JjaC5zdGFjayhbei5zb2Z0bWF4KDEpIGZvciB6IGluIGxvZ2l0c10pLm1lYW4oMCkKICAgIGVsaWYgc3BhY2UgPT0gImxvZ2l0IjoKICAgICAgICBwcm9icyA9IHRvcmNoLnN0YWNrKGxvZ2l0cykubWVhbigwKS5zb2Z0bWF4KDEpCiAgICBlbHNlOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoInNwYWNlIG11c3QgYmUgcHJvYiBvciBsb2dpdCIpCiAgICBpZiB0ZW1wZXJhdHVyZSBpcyBub3QgTm9uZToKICAgICAgICBwcm9icyA9IChwcm9icy5jbGFtcF9taW4oMWUtMTIpLmxvZygpIC8gdGVtcGVyYXR1cmUpLnNvZnRtYXgoMSkKICAgIHJldHVybiAocHJvYnMsIHRvcmNoLnN0YWNrKGxvZ2l0cykpIGlmIHJldHVybl9sb2dpdHMgZWxzZSBwcm9icwoKCmRlZiBwcmVkaWN0X21ldGhvZChtb2RlbCwgbG9hZGVyLCBkZXZpY2UsIG1ldGhvZD0iSTAwIiwgc3BhY2U9InByb2IiLCBjcm9wPU5vbmUsIGxvZ2l0c19wYXRoPU5vbmUpOgogICAgaW1wb3J0IG51bXB5IGFzIG5wCiAgICBpbXBvcnQgdG9yY2gKICAgIGZyb20gcnVudGltZSBpbXBvcnQgY2hlY2tfbWVtb3J5LCByZXNvdXJjZV9yZXBvcnQKICAgIG1vZGVsLmV2YWwoKQogICAgbmFtZXMsIGxhYmVscywgcHJlZGljdGlvbnMsIHZpZXdfbG9naXRzID0gW10sIFtdLCBbXSwgW10KICAgIHdpdGggdG9yY2guaW5mZXJlbmNlX21vZGUoKToKICAgICAgICBmb3Igc3RlcCwgKGltYWdlcywgdGFyZ2V0LCBiYXRjaF9uYW1lcykgaW4gZW51bWVyYXRlKGxvYWRlciwgMSk6CiAgICAgICAgICAgIGlmIHN0ZXAgPT0gMSBvciBzdGVwICUgNTAgPT0gMDoKICAgICAgICAgICAgICAgIGNoZWNrX21lbW9yeSgpCiAgICAgICAgICAgIHJlc3VsdCA9IGZvcndhcmRfcHJvYmFiaWxpdGllcyhtb2RlbCwgaW1hZ2VzLnRvKGRldmljZSwgbm9uX2Jsb2NraW5nPVRydWUpLCBtZXRob2QsIHNwYWNlLCBjcm9wLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgcmV0dXJuX2xvZ2l0cz1sb2dpdHNfcGF0aCBpcyBub3QgTm9uZSkKICAgICAgICAgICAgaWYgbG9naXRzX3BhdGggaXMgbm90IE5vbmU6CiAgICAgICAgICAgICAgICBwcm9icywgbG9naXRzID0gcmVzdWx0CiAgICAgICAgICAgICAgICB2aWV3X2xvZ2l0cy5hcHBlbmQobG9naXRzLmNwdSgpLm51bXB5KCkpICAjIEsgeCBiYXRjaCB4IDksIG5vIGZlYXR1cmUgdGVuc29ycy4KICAgICAgICAgICAgZWxzZToKICAgICAgICAgICAgICAgIHByb2JzID0gcmVzdWx0CiAgICAgICAgICAgIHByZWRpY3Rpb25zLmFwcGVuZChwcm9icy5jcHUoKS5udW1weSgpKQogICAgICAgICAgICBsYWJlbHMuYXBwZW5kKHRhcmdldC5udW1weSgpKQogICAgICAgICAgICBuYW1lcy5leHRlbmQoYmF0Y2hfbmFtZXMpCiAgICAgICAgICAgIGlmIHN0ZXAgPT0gMSBvciBzdGVwID09IGxlbihsb2FkZXIpOgogICAgICAgICAgICAgICAgcHJpbnQoZiJ7bWV0aG9kfSBpbmZlcmVuY2Uge3N0ZXB9L3tsZW4obG9hZGVyKX0gfCB7cmVzb3VyY2VfcmVwb3J0KCl9IiwgZmx1c2g9VHJ1ZSkKICAgIGlmIG5vdCBuYW1lczoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJDYW5ub3QgcHJlZGljdCBhbiBlbXB0eSBsb2FkZXIiKQogICAgaWYgbG9naXRzX3BhdGggaXMgbm90IE5vbmU6CiAgICAgICAgcmF3ID0gbnAuY29uY2F0ZW5hdGUodmlld19sb2dpdHMsIGF4aXM9MSkKICAgICAgICBucC5zYXZlKGxvZ2l0c19wYXRoLCByYXdbMF0gaWYgbGVuKHJhdykgPT0gMSBlbHNlIHJhdykKICAgIHJldHVybiBuYW1lcywgbnAuY29uY2F0ZW5hdGUobGFiZWxzKSwgbnAuY29uY2F0ZW5hdGUocHJlZGljdGlvbnMpCgoKZGVmIHByZWRpY3RfbG9naXRzKG1vZGVsLCBsb2FkZXIsIGRldmljZSwgdmlldz1Ob25lLCBhdXRvY2FzdDogYm9vbCA9IEZhbHNlKToKICAgIGltcG9ydCBudW1weSBhcyBucAogICAgaW1wb3J0IHRvcmNoCgogICAgbW9kZWwuZXZhbCgpCiAgICBmaWxlbmFtZXMsIGxhYmVscywgbG9naXRzX2FsbCA9IFtdLCBbXSwgW10KICAgIHVzZV9hbXAgPSBib29sKGF1dG9jYXN0IGFuZCB0b3JjaC5kZXZpY2UoZGV2aWNlKS50eXBlID09ICJjdWRhIikKICAgIHdpdGggdG9yY2guaW5mZXJlbmNlX21vZGUoKToKICAgICAgICBmb3IgaW1hZ2VzLCB0YXJnZXQsIG5hbWVzIGluIGxvYWRlcjoKICAgICAgICAgICAgaW1hZ2VzID0gaW1hZ2VzLnRvKGRldmljZSwgbm9uX2Jsb2NraW5nPVRydWUpCiAgICAgICAgICAgIGlmIHZpZXcgaXMgbm90IE5vbmU6CiAgICAgICAgICAgICAgICBpbWFnZXMgPSB2aWV3KGltYWdlcykKICAgICAgICAgICAgd2l0aCB0b3JjaC5hdXRvY2FzdChkZXZpY2VfdHlwZT10b3JjaC5kZXZpY2UoZGV2aWNlKS50eXBlLCBkdHlwZT10b3JjaC5mbG9hdDE2LCBlbmFibGVkPXVzZV9hbXApOgogICAgICAgICAgICAgICAgbG9naXRzID0gbW9kZWwoaW1hZ2VzKQogICAgICAgICAgICBsb2dpdHNfYWxsLmFwcGVuZChsb2dpdHMuZmxvYXQoKS5jcHUoKS5udW1weSgpKQogICAgICAgICAgICBsYWJlbHMuYXBwZW5kKHRhcmdldC5jcHUoKS5udW1weSgpKQogICAgICAgICAgICBmaWxlbmFtZXMuZXh0ZW5kKGxpc3QobmFtZXMpKQogICAgaWYgbm90IGxvZ2l0c19hbGw6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiQ2Fubm90IHByZWRpY3QgZnJvbSBhbiBlbXB0eSBsb2FkZXIiKQogICAgcmV0dXJuIGZpbGVuYW1lcywgbnAuY29uY2F0ZW5hdGUobGFiZWxzKSwgbnAuY29uY2F0ZW5hdGUobG9naXRzX2FsbCkKCgpkZWYgcHJlZGljdF9tdWx0aWNyb3AobW9kZWwsIGxvYWRlciwgZGV2aWNlLCBjcm9wOiBpbnQsIHNwYWNlOiBzdHIgPSAicHJvYiIpOgogICAgIiIiRXZhbHVhdGUgZml2ZSBkZXRlcm1pbmlzdGljIGNyb3BzIGFuZCByZXR1cm4gYWdncmVnYXRlZCBwcm9iYWJpbGl0aWVzLiIiIgogICAgaW1wb3J0IG51bXB5IGFzIG5wCiAgICBpbXBvcnQgdG9yY2gKICAgIGltcG9ydCB0b3JjaC5ubi5mdW5jdGlvbmFsIGFzIEYKCiAgICBtb2RlbC5ldmFsKCkKICAgIGZpbGVuYW1lcywgbGFiZWxzLCBwcm9ic19hbGwgPSBbXSwgW10sIFtdCiAgICB3aXRoIHRvcmNoLmluZmVyZW5jZV9tb2RlKCk6CiAgICAgICAgZm9yIGltYWdlcywgdGFyZ2V0LCBuYW1lcyBpbiBsb2FkZXI6CiAgICAgICAgICAgIGltYWdlcyA9IGltYWdlcy50byhkZXZpY2UsIG5vbl9ibG9ja2luZz1UcnVlKQogICAgICAgICAgICB2aWV3cyA9IHZpZXdzX211bHRpY3JvcChpbWFnZXMsIGNyb3ApCiAgICAgICAgICAgIHZpZXdzID0gW0YuaW50ZXJwb2xhdGUodmlldywgc2l6ZT1pbWFnZXMuc2hhcGVbLTI6XSwgbW9kZT0iYmlsaW5lYXIiLCBhbGlnbl9jb3JuZXJzPUZhbHNlKQogICAgICAgICAgICAgICAgICAgICBmb3IgdmlldyBpbiB2aWV3c10KICAgICAgICAgICAgbG9naXRzID0gW21vZGVsKHZpZXcpIGZvciB2aWV3IGluIHZpZXdzXQogICAgICAgICAgICBwcm9ic19hbGwuYXBwZW5kKGFnZ3JlZ2F0ZV92aWV3cyhsb2dpdHMsIHNwYWNlKSkKICAgICAgICAgICAgbGFiZWxzLmFwcGVuZCh0YXJnZXQuY3B1KCkubnVtcHkoKSkKICAgICAgICAgICAgZmlsZW5hbWVzLmV4dGVuZChsaXN0KG5hbWVzKSkKICAgIGlmIG5vdCBwcm9ic19hbGw6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiQ2Fubm90IHByZWRpY3QgZnJvbSBhbiBlbXB0eSBsb2FkZXIiKQogICAgcmV0dXJuIGZpbGVuYW1lcywgbnAuY29uY2F0ZW5hdGUobGFiZWxzKSwgbnAuY29uY2F0ZW5hdGUocHJvYnNfYWxsKQoKCmRlZiB2aWV3X2lkZW50aXR5KHgpOgogICAgcmV0dXJuIHgKCgpkZWYgdmlld19oZmxpcCh4KToKICAgIGltcG9ydCB0b3JjaAogICAgcmV0dXJuIHRvcmNoLmZsaXAoeCwgZGltcz0oLTEsKSkKCgpkZWYgdmlld3NfbXVsdGljcm9wKHgsIGNyb3A6IGludCk6CiAgICBpZiB4Lm5kaW0gIT0gNDoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJ4IG11c3QgaGF2ZSBzaGFwZSBOQ0hXIikKICAgIGhlaWdodCwgd2lkdGggPSB4LnNoYXBlWy0yOl0KICAgIGlmIGNyb3AgPCAxIG9yIGNyb3AgPiBtaW4oaGVpZ2h0LCB3aWR0aCk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmImNyb3AgbXVzdCBiZSBiZXR3ZWVuIDEgYW5kIHttaW4oaGVpZ2h0LCB3aWR0aCl9IikKICAgIHBvc2l0aW9ucyA9IFsoMCwgMCksICgwLCB3aWR0aCAtIGNyb3ApLCAoaGVpZ2h0IC0gY3JvcCwgMCksCiAgICAgICAgICAgICAgICAgKGhlaWdodCAtIGNyb3AsIHdpZHRoIC0gY3JvcCksICgoaGVpZ2h0IC0gY3JvcCkgLy8gMiwgKHdpZHRoIC0gY3JvcCkgLy8gMildCiAgICByZXR1cm4gW3hbOiwgOiwgdG9wOnRvcCArIGNyb3AsIGxlZnQ6bGVmdCArIGNyb3BdIGZvciB0b3AsIGxlZnQgaW4gcG9zaXRpb25zXQoKCmRlZiB2aWV3c19tdWx0aXNjYWxlKHgsIHNpemVzKToKICAgIGltcG9ydCB0b3JjaC5ubi5mdW5jdGlvbmFsIGFzIEYKICAgIHNpemVzID0gW2ludChzaXplKSBmb3Igc2l6ZSBpbiBzaXplc10KICAgIGlmIHgubmRpbSAhPSA0IG9yIG5vdCBzaXplcyBvciBhbnkoc2l6ZSA8IDEgZm9yIHNpemUgaW4gc2l6ZXMpOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoInggbXVzdCBiZSBOQ0hXIGFuZCBzaXplcyBtdXN0IGNvbnRhaW4gcG9zaXRpdmUgZGltZW5zaW9ucyIpCiAgICByZXR1cm4gW0YuaW50ZXJwb2xhdGUoeCwgc2l6ZT0oc2l6ZSwgc2l6ZSksIG1vZGU9ImJpbGluZWFyIiwgYWxpZ25fY29ybmVycz1GYWxzZSkKICAgICAgICAgICAgZm9yIHNpemUgaW4gc2l6ZXNdCgoKZGVmIGFnZ3JlZ2F0ZV92aWV3cyhsb2dpdHNfcGVyX3ZpZXcsIHNwYWNlOiBzdHIgPSAicHJvYiIpOgogICAgaW1wb3J0IHRvcmNoCiAgICBpbXBvcnQgdG9yY2gubm4uZnVuY3Rpb25hbCBhcyBGCgogICAgaWYgbm90IGxvZ2l0c19wZXJfdmlldzoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJBdCBsZWFzdCBvbmUgdmlldyBpcyByZXF1aXJlZCIpCiAgICB0ZW5zb3JzID0gW3RvcmNoLmFzX3RlbnNvcih2YWx1ZSwgZHR5cGU9dG9yY2guZmxvYXQzMikgZm9yIHZhbHVlIGluIGxvZ2l0c19wZXJfdmlld10KICAgIGlmIGFueSh2YWx1ZS5zaGFwZSAhPSB0ZW5zb3JzWzBdLnNoYXBlIGZvciB2YWx1ZSBpbiB0ZW5zb3JzKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJBbGwgdmlldyBsb2dpdHMgbXVzdCBoYXZlIHRoZSBzYW1lIHNoYXBlIikKICAgIGlmIHNwYWNlID09ICJwcm9iIjoKICAgICAgICBwcm9icyA9IHRvcmNoLnN0YWNrKFtGLnNvZnRtYXgodmFsdWUsIGRpbT0xKSBmb3IgdmFsdWUgaW4gdGVuc29yc10pLm1lYW4oZGltPTApCiAgICBlbGlmIHNwYWNlID09ICJsb2dpdCI6CiAgICAgICAgcHJvYnMgPSBGLnNvZnRtYXgodG9yY2guc3RhY2sodGVuc29ycykubWVhbihkaW09MCksIGRpbT0xKQogICAgZWxzZToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJzcGFjZSBtdXN0IGJlIHByb2Igb3IgbG9naXQiKQogICAgcmV0dXJuIHByb2JzLmNwdSgpLm51bXB5KCkKCgpkZWYgZW5zZW1ibGVfcHJvYnMobGlzdF9vZl9wcm9icyk6CiAgICBpbXBvcnQgbnVtcHkgYXMgbnAKCiAgICBhcnJheXMgPSBbbnAuYXNhcnJheSh2YWx1ZSwgZHR5cGU9bnAuZmxvYXQ2NCkgZm9yIHZhbHVlIGluIGxpc3Rfb2ZfcHJvYnNdCiAgICBpZiBub3QgYXJyYXlzOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIkF0IGxlYXN0IG9uZSBwcm9iYWJpbGl0eSBtYXRyaXggaXMgcmVxdWlyZWQiKQogICAgaWYgYW55KGFycmF5LnNoYXBlICE9IGFycmF5c1swXS5zaGFwZSBmb3IgYXJyYXkgaW4gYXJyYXlzKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJBbGwgcHJvYmFiaWxpdHkgbWF0cmljZXMgbXVzdCBoYXZlIHRoZSBzYW1lIHNoYXBlIGFuZCByb3cgb3JkZXIiKQogICAgaWYgYW55KG5wLmFueShhcnJheSA8IDApIG9yIG5vdCBucC5hbGxjbG9zZShhcnJheS5zdW0oYXhpcz0xKSwgMS4wLCBhdG9sPTFlLTUpIGZvciBhcnJheSBpbiBhcnJheXMpOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIkVhY2ggaW5wdXQgbXVzdCBjb250YWluIG5vcm1hbGl6ZWQgcHJvYmFiaWxpdGllcyIpCiAgICByZXR1cm4gbnAubWVhbihhcnJheXMsIGF4aXM9MCkKCgpkZWYgZml0X3RlbXBlcmF0dXJlKHZhbF9sb2dpdHMsIHZhbF9sYWJlbHMpIC0+IGZsb2F0OgogICAgaW1wb3J0IHRvcmNoCiAgICBpbXBvcnQgdG9yY2gubm4uZnVuY3Rpb25hbCBhcyBGCgogICAgbG9naXRzID0gdG9yY2guYXNfdGVuc29yKHZhbF9sb2dpdHMsIGR0eXBlPXRvcmNoLmZsb2F0NjQsIGRldmljZT0iY3B1IikKICAgIGxhYmVscyA9IHRvcmNoLnRlbnNvcih2YWxfbGFiZWxzLCBkdHlwZT10b3JjaC5sb25nLCBkZXZpY2U9ImNwdSIpCiAgICBpZiBsb2dpdHMubmRpbSAhPSAyIG9yIGxlbihsYWJlbHMpICE9IGxlbihsb2dpdHMpIG9yIGxlbihsYWJlbHMpID09IDA6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigidmFsX2xvZ2l0cyBtdXN0IGJlIChOLEspLCB3aXRoIG9uZSBsYWJlbCBwZXIgcm93IikKICAgIGxvZ190ID0gdG9yY2gubm4uUGFyYW1ldGVyKHRvcmNoLnplcm9zKCgpLCBkdHlwZT10b3JjaC5mbG9hdDY0KSkKICAgIG9wdGltaXplciA9IHRvcmNoLm9wdGltLkxCRkdTKFtsb2dfdF0sIGxyPTAuMSwgbWF4X2l0ZXI9MTAwLCBsaW5lX3NlYXJjaF9mbj0ic3Ryb25nX3dvbGZlIikKCiAgICBkZWYgY2xvc3VyZSgpOgogICAgICAgIG9wdGltaXplci56ZXJvX2dyYWQoKQogICAgICAgIGxvc3MgPSBGLmNyb3NzX2VudHJvcHkobG9naXRzIC8gbG9nX3QuY2xhbXAoLTQuMCwgNC4wKS5leHAoKSwgbGFiZWxzKQogICAgICAgIGxvc3MuYmFja3dhcmQoKQogICAgICAgIHJldHVybiBsb3NzCgogICAgb3B0aW1pemVyLnN0ZXAoY2xvc3VyZSkKICAgIHJldHVybiBmbG9hdChsb2dfdC5kZXRhY2goKS5jbGFtcCgtNC4wLCA0LjApLmV4cCgpLml0ZW0oKSkKCgpkZWYgYXBwbHlfdGVtcGVyYXR1cmUobG9naXRzLCBUOiBmbG9hdCk6CiAgICBpbXBvcnQgbnVtcHkgYXMgbnAKICAgIGltcG9ydCB0b3JjaAogICAgaW1wb3J0IHRvcmNoLm5uLmZ1bmN0aW9uYWwgYXMgRgoKICAgIGlmIGZsb2F0KFQpIDw9IDA6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiVCBtdXN0IGJlIHBvc2l0aXZlIikKICAgIHZhbHVlcyA9IHRvcmNoLmFzX3RlbnNvcihsb2dpdHMsIGR0eXBlPXRvcmNoLmZsb2F0NjQpCiAgICByZXR1cm4gRi5zb2Z0bWF4KHZhbHVlcyAvIGZsb2F0KFQpLCBkaW09MSkuY3B1KCkubnVtcHkoKS5hc3R5cGUobnAuZmxvYXQ2NCkKCgpkZWYgZnVzZV9jb252X2JuKG1vZGVsKToKICAgICIiIlJldHVybiBhbiBldmFsIGNvcHkgd2l0aCBhZGphY2VudCBDb252MmQvQmF0Y2hOb3JtMmQgcGFpcnMgZnVzZWQuIiIiCiAgICBpbXBvcnQgY29weQogICAgaW1wb3J0IHRvcmNoCiAgICBmcm9tIHRvcmNoIGltcG9ydCBubgogICAgZnJvbSB0b3JjaC5ubi51dGlscy5mdXNpb24gaW1wb3J0IGZ1c2VfY29udl9ibl9ldmFsCgogICAgZnVzZWQgPSBjb3B5LmRlZXBjb3B5KG1vZGVsKS5ldmFsKCkKICAgIHBhaXJzID0gWzBdCgogICAgZGVmIGZvbGRfY2hpbGRyZW4ocGFyZW50KToKICAgICAgICBmb3IgY2hpbGQgaW4gbGlzdChwYXJlbnQuY2hpbGRyZW4oKSk6CiAgICAgICAgICAgIGZvbGRfY2hpbGRyZW4oY2hpbGQpCiAgICAgICAgIyBSZWdpc3RyYXRpb24gb3JkZXIgaW4gYW4gYXJiaXRyYXJ5IG5uLk1vZHVsZSBpcyBOT1QgZXhlY3V0aW9uIG9yZGVyLgogICAgICAgICMgT25seSBTZXF1ZW50aWFsIGd1YXJhbnRlZXMgQ29udiAtPiBCTiBhZGphY2VuY3k7IGxlYXZlIG90aGVyIGdyYXBocyB1bmNoYW5nZWQuCiAgICAgICAgaWYgbm90IGlzaW5zdGFuY2UocGFyZW50LCBubi5TZXF1ZW50aWFsKToKICAgICAgICAgICAgcmV0dXJuCiAgICAgICAgY2hpbGRyZW4gPSBsaXN0KHBhcmVudC5uYW1lZF9jaGlsZHJlbigpKQogICAgICAgIGZvciAobGVmdF9uYW1lLCBsZWZ0KSwgKHJpZ2h0X25hbWUsIHJpZ2h0KSBpbiB6aXAoY2hpbGRyZW4sIGNoaWxkcmVuWzE6XSk6CiAgICAgICAgICAgIGlmIGlzaW5zdGFuY2UobGVmdCwgbm4uQ29udjJkKSBhbmQgaXNpbnN0YW5jZShyaWdodCwgbm4uQmF0Y2hOb3JtMmQpOgogICAgICAgICAgICAgICAgc2V0YXR0cihwYXJlbnQsIGxlZnRfbmFtZSwgZnVzZV9jb252X2JuX2V2YWwobGVmdCwgcmlnaHQpKQogICAgICAgICAgICAgICAgc2V0YXR0cihwYXJlbnQsIHJpZ2h0X25hbWUsIG5uLklkZW50aXR5KCkpCiAgICAgICAgICAgICAgICBwYWlyc1swXSArPSAxCgogICAgZm9sZF9jaGlsZHJlbihmdXNlZCkKICAgIGlmIG5vdCBwYWlyc1swXToKICAgICAgICBmdXNlZC5ibl9mdXNpb25fZXJyb3IgPSBOb25lCiAgICAgICAgZnVzZWQuYm5fZnVzZWRfcGFpcnMgPSAwCiAgICAgICAgcmV0dXJuIGZ1c2VkCgogICAgcGFyYW1ldGVyID0gbmV4dChmdXNlZC5wYXJhbWV0ZXJzKCkpCiAgICBjZmcgPSBnZXRhdHRyKGZ1c2VkLCAicHJldHJhaW5lZF9jZmciLCBOb25lKSBvciBnZXRhdHRyKGZ1c2VkLCAiZGVmYXVsdF9jZmciLCB7fSkgb3Ige30KICAgIGlucHV0X3NpemUgPSBjZmcuZ2V0KCJpbnB1dF9zaXplIiwgKDMsIDIyNCwgMjI0KSkgaWYgaXNpbnN0YW5jZShjZmcsIGRpY3QpIGVsc2UgKDMsIDIyNCwgMjI0KQogICAgc2l6ZSA9IGludChpbnB1dF9zaXplWy0xXSkKICAgIHNhbXBsZSA9IHRvcmNoLnplcm9zKDEsIDMsIHNpemUsIHNpemUsIGRldmljZT1wYXJhbWV0ZXIuZGV2aWNlLCBkdHlwZT1wYXJhbWV0ZXIuZHR5cGUpCiAgICB0cmFpbmluZ19zdGF0ZXMgPSBbKG1vZHVsZSwgbW9kdWxlLnRyYWluaW5nKSBmb3IgbW9kdWxlIGluIG1vZGVsLm1vZHVsZXMoKV0KICAgIG1vZGVsLmV2YWwoKQogICAgdHJ5OgogICAgICAgIHdpdGggdG9yY2guaW5mZXJlbmNlX21vZGUoKToKICAgICAgICAgICAgZXhwZWN0ZWQgPSBtb2RlbChzYW1wbGUpCiAgICAgICAgICAgIGFjdHVhbCA9IGZ1c2VkKHNhbXBsZSkKICAgIGZpbmFsbHk6CiAgICAgICAgZm9yIG1vZHVsZSwgd2FzX3RyYWluaW5nIGluIHRyYWluaW5nX3N0YXRlczoKICAgICAgICAgICAgbW9kdWxlLnRyYWluaW5nID0gd2FzX3RyYWluaW5nCiAgICBmdXNlZC5ibl9mdXNpb25fZXJyb3IgPSBmbG9hdCgoZXhwZWN0ZWQgLSBhY3R1YWwpLmFicygpLm1heCgpLml0ZW0oKSkKICAgIGZ1c2VkLmJuX2Z1c2VkX3BhaXJzID0gcGFpcnNbMF0KICAgIHJldHVybiBmdXNlZAo=",
  "train.py": "...",
  "runner.py": "IiIiUnVuIGVhY2ggZXhwZXJpbWVudCBpbiBhIGZyZXNoIFB5dGhvbiBwcm9jZXNzIGFuZCBzdHJlYW0gZHVyYWJsZSB0ZXh0IGxvZ3MuIiIiCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBoYXNobGliCmltcG9ydCBqc29uCmltcG9ydCBvcwppbXBvcnQgc3VicHJvY2VzcwppbXBvcnQgc3lzCmZyb20gZGF0YWNsYXNzZXMgaW1wb3J0IGFzZGljdApmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKCmZyb20gcnVudGltZSBpbXBvcnQgYXRvbWljX2pzb24sIGNvbmZpZ3VyZV9lbnZpcm9ubWVudAoKCmRlZiBsYXVuY2goY2ZnKToKICAgIGltcG9ydCB0cmFpbgogICAgY29uZmlndXJlX2Vudmlyb25tZW50KCkKICAgIGZvbGRlciA9IHRyYWluLnJ1bl9kaXIoY2ZnKQogICAgZm9sZGVyLm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgIHJlc3VsdF9wYXRoID0gZm9sZGVyIC8gInJlc3VsdC5qc29uIgogICAgY29uZmlnX3BhdGggPSBmb2xkZXIgLyAiY29uZmlnLmpzb24iCiAgICBlbnZfcGF0aCA9IGZvbGRlciAvICJlbnZpcm9ubWVudC5qc29uIgogICAgaWYgcmVzdWx0X3BhdGguZXhpc3RzKCkgYW5kIGNvbmZpZ19wYXRoLmV4aXN0cygpIGFuZCBlbnZfcGF0aC5leGlzdHMoKToKICAgICAgICByZXN1bHQgPSBqc29uLmxvYWRzKHJlc3VsdF9wYXRoLnJlYWRfdGV4dChlbmNvZGluZz0idXRmLTgiKSkKICAgICAgICBzYXZlZCA9IGpzb24ubG9hZHMoY29uZmlnX3BhdGgucmVhZF90ZXh0KGVuY29kaW5nPSJ1dGYtOCIpKQogICAgICAgIGVudiA9IGpzb24ubG9hZHMoZW52X3BhdGgucmVhZF90ZXh0KGVuY29kaW5nPSJ1dGYtOCIpKQogICAgICAgIGlmIG5vdCB0cmFpbi5fc2FtZV90cmFpbmluZ19jb25maWcoc2F2ZWQsIGNmZyk6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJTYXZlZCBjb25maWcgZGlmZmVycyBmb3Ige2ZvbGRlcn07IGNob29zZSBhIG5ldyBvdXRwdXQgZGlyZWN0b3J5IikKICAgICAgICBpZiBlbnYuZ2V0KCJzcGxpdF9zaGEyNTYiKSAhPSB0cmFpbi5zcGxpdF9maW5nZXJwcmludChjZmcpOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJTcGxpdCBDU1YgY2hhbmdlZDsgY2Fubm90IHJldXNlIHJlc3VsdHMiKQogICAgICAgIHNvdXJjZXMgPSB7cC5uYW1lOmhhc2hsaWIuc2hhMjU2KHAucmVhZF9ieXRlcygpKS5oZXhkaWdlc3QoKSBmb3IgcCBpbiBQYXRoKF9fZmlsZV9fKS5wYXJlbnQuZ2xvYigiKi5weSIpfQogICAgICAgIGlmIGVudi5nZXQoInNvdXJjZV9zaGEyNTYiKSAhPSBzb3VyY2VzOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJTb3VyY2UgY2hhbmdlZCBzaW5jZSB0aGlzIGV4cGVyaW1lbnQ7IHByZXNlcnZlIG9sZCBvdXRwdXQgYW5kIGNob29zZSBhIG5ldyBvdXRwdXQgZGlyZWN0b3J5IikKICAgICAgICBmaW5pc2hlZCA9IHJlc3VsdC5nZXQoImVwb2Noc19jb21wbGV0ZWQiLCAwKSA9PSBjZmcuZXBvY2hzCiAgICAgICAgZm9yIGtleSwgdGFyZ2V0IGluICgoImNoZWNrcG9pbnQiLCBmb2xkZXIgLyAiY2hlY2twb2ludF9iZXN0LnB0IiksCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAoImN1cnZlIiwgUGF0aChjZmcuY3VydmVzX2RpcikvZiJ7Y2ZnLmV4cF9pZH1fc2VlZHtjZmcuc2VlZH0ucG5nIiksCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAoInZhbF9wcmVkaWN0aW9ucyIsIHRyYWluLnByZWRfcGF0aChjZmcsInZhbCIpKSk6CiAgICAgICAgICAgIHJlc3VsdFtrZXldID0gc3RyKHRhcmdldCkKICAgICAgICAgICAgZmluaXNoZWQgPSBmaW5pc2hlZCBhbmQgdGFyZ2V0LmlzX2ZpbGUoKQogICAgICAgIGlmIGNmZy5zYXZlX3Rlc3RfcHJlZGljdGlvbnM6CiAgICAgICAgICAgIGZpbmlzaGVkID0gZmluaXNoZWQgYW5kIHRyYWluLnByZWRfcGF0aChjZmcsInRlc3QiKS5pc19maWxlKCkgYW5kIGJvb2wocmVzdWx0LmdldCgidGVzdF9tZXRyaWNzIikpCiAgICAgICAgICAgIGZvciBrZXkgaW4gKCJpbmZlcmVuY2VfbWV0aG9kIiwgImFnZ3JlZ2F0ZV9zcGFjZSIsICJ0ZW1wZXJhdHVyZV9zY2FsZSIpOgogICAgICAgICAgICAgICAgaWYgdHJhaW4ucHJlZF9wYXRoKGNmZywidGVzdCIpLmV4aXN0cygpIGFuZCBzYXZlZC5nZXQoa2V5KSAhPSBnZXRhdHRyKGNmZyxrZXkpOgogICAgICAgICAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIkZpbmFsIGluZmVyZW5jZSBjb25maWd1cmF0aW9uIGFscmVhZHkgbG9ja2VkIGJ5IHNhdmVkIHRlc3QgcHJlZGljdGlvbnMiKQogICAgICAgIGlmIGZpbmlzaGVkOgogICAgICAgICAgICBwcmludChmIlJldXNlIHtjZmcuZXhwX2lkfS9zZWVke2NmZy5zZWVkfTogY29tcGxldGVkIGFydGlmYWN0cyB2ZXJpZmllZCIsIGZsdXNoPVRydWUpCiAgICAgICAgICAgIHJldHVybiByZXN1bHQKICAgIHJlcXVlc3QgPSBmb2xkZXIgLyAicmVxdWVzdC5qc29uIgogICAgYXRvbWljX2pzb24ocmVxdWVzdCwgYXNkaWN0KGNmZykpCiAgICBjb21tYW5kID0gW3N5cy5leGVjdXRhYmxlLCAiLXUiLCBzdHIoUGF0aChfX2ZpbGVfXykud2l0aF9uYW1lKCJ0cmFpbi5weSIpKSwgIi0tY29uZmlnIiwgc3RyKHJlcXVlc3QucmVzb2x2ZSgpKV0KICAgIHByaW50KGYiU3RhcnQgd29ya2VyIHtjZmcuZXhwX2lkfS9zZWVke2NmZy5zZWVkfTsgbG9nPXtmb2xkZXIgLyAncnVuLmxvZyd9IiwgZmx1c2g9VHJ1ZSkKICAgICMgTmV2ZXIgYnVmZmVyIHRoZSB3aG9sZSBzdGRvdXQgaW4gUkFNLiBFYWNoIGxpbmUgaXMgcGVyc2lzdGVkIGJlZm9yZSBkaXNwbGF5LgogICAgd2l0aCAoZm9sZGVyIC8gInJ1bi5sb2ciKS5vcGVuKCJhIiwgZW5jb2Rpbmc9InV0Zi04IiwgYnVmZmVyaW5nPTEpIGFzIGxvZzoKICAgICAgICBwcm9jZXNzID0gc3VicHJvY2Vzcy5Qb3Blbihjb21tYW5kLCBzdGRvdXQ9c3VicHJvY2Vzcy5QSVBFLCBzdGRlcnI9c3VicHJvY2Vzcy5TVERPVVQsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgdGV4dD1UcnVlLCBlbmNvZGluZz0idXRmLTgiLCBlcnJvcnM9InJlcGxhY2UiLCBidWZzaXplPTEsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgZW52PXsqKm9zLmVudmlyb24sICJQWVRIT05VTkJVRkZFUkVEIjoiMSIsICJQWVRIT05JT0VOQ09ESU5HIjoidXRmLTgifSkKICAgICAgICB0cnk6CiAgICAgICAgICAgIGZvciBsaW5lIGluIHByb2Nlc3Muc3Rkb3V0OgogICAgICAgICAgICAgICAgbG9nLndyaXRlKGxpbmUpCiAgICAgICAgICAgICAgICBwcmludChsaW5lLCBlbmQ9IiIsIGZsdXNoPVRydWUpCiAgICAgICAgICAgIHN0YXR1cyA9IHByb2Nlc3Mud2FpdCgpCiAgICAgICAgZXhjZXB0IEJhc2VFeGNlcHRpb246CiAgICAgICAgICAgIHByb2Nlc3MudGVybWluYXRlKCkKICAgICAgICAgICAgcHJvY2Vzcy53YWl0KCkKICAgICAgICAgICAgcmFpc2UKICAgICAgICBmaW5hbGx5OgogICAgICAgICAgICBwcm9jZXNzLnN0ZG91dC5jbG9zZSgpCiAgICBpZiBzdGF0dXM6CiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKGYiV29ya2VyIHtjZmcuZXhwX2lkfSBleGl0ZWQgd2l0aCB7c3RhdHVzfS4gU2VlIHtmb2xkZXIgLyAncnVuLmxvZyd9LiAiCiAgICAgICAgICAgICAgICAgICAgICAgICAgICJSZXJ1biB3aXRoIHRoZSBzYW1lIHNldHRpbmdzIHRvIHJlc3VtZSB0aGUgbGFzdCBjb21wbGV0ZWQgZXBvY2guIikKICAgIGlmIG5vdCByZXN1bHRfcGF0aC5pc19maWxlKCk6CiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCJXb3JrZXIgcmV0dXJuZWQgd2l0aG91dCByZXN1bHQuanNvbiIpCiAgICByZXR1cm4ganNvbi5sb2FkcyhyZXN1bHRfcGF0aC5yZWFkX3RleHQoZW5jb2Rpbmc9InV0Zi04IikpCg==",
  "eval.py": "..."
}
for name, encoded in EMBEDDED_SOURCE.items():
    (CODE_DIR / name).write_bytes(base64.b64decode(encoded))
print(f'Wrote {len(EMBEDDED_SOURCE)} modules to {CODE_DIR}', flush=True)


In [ ]:
import hashlib
import sys
EXPECTED_SOURCE_SHA256 = {
  "runtime.py": "67973befbb3dcb65742ebab6db1177c137cb1f02fb10591d41661edbaf15530d",
  "dataset.py": "85ad26bf6e5b8ce2f5301a72ff49bac2dd8631937b60423f183267c8002b274f",
  "model.py": "3959d96824cc3a26008ef7bbeda5c8f59eb52a3c746e49bf97bcb0d263fa3c75",
  "losses.py": "110ced89038de0a1e91433ed2b95b4c17dd2b83cdde297ae753b186d48e14ade",
  "benchmark.py": "3d8010f279b45279f3592ecd5d5f6ad424a51fde029e5b3a99a787075ad583b0",
  "inference.py": "7ddb0fcd2773522140a539dc8e94c1c43ab4787e0f521bf69b693dec2244a14e",
  "train.py": "aaa853dbecb709fe44811fce938b61bf8d884be5dac2c4a6c20a706139270ecf",
  "runner.py": "e733a68d16e71ee68059707eb419d2e06ca92923bfe792138cdd2ee49b23a5c8",
  "eval.py": "1210f3c7ca1cb9b4488383f1ceeaa139bfdf5c683694b300f6c55b6b9cdde28e"
}
for name, expected in EXPECTED_SOURCE_SHA256.items():
    actual = hashlib.sha256((CODE_DIR / name).read_bytes()).hexdigest()
    if actual != expected:
        raise RuntimeError(f"Source mismatch: {name}")
sys.path.insert(0, str(CODE_DIR))
for name in ("dataset","model","losses","benchmark","inference","runtime","runner","train","eval"):
    sys.modules.pop(name, None)
import dataset
import model as model_utils
import losses
import benchmark
import inference
import train
import runner
import eval as eval_utils
from runtime import atomic_json
import numpy as np
import pandas as pd
from tqdm import tqdm
print("Source SHA256 khớp tất cả module; eval.py là bản gốc.", flush=True)


## Dùng Dataset đã gắn vào Kaggle

Notebook chọn Dataset trong `/kaggle/input` có đủ CSV fold 0. Với cấu trúc `data/labels` và `data/images`, nhãn và ảnh được đọc trực tiếp từ Dataset; cell in ra đường dẫn đang dùng. Nếu có nhiều Dataset phù hợp, điền tên thư mục Dataset vào `DATASET_SLUG` trong cấu hình. Chỉ khi ảnh nằm trong `images.zip` hoặc thư mục lồng sâu, notebook mới giải nén/liên kết ảnh vào `/kaggle/working`.

In [ ]:
import hashlib
import shutil
import zipfile

def find_dataset_and_labels():
    if not INPUT_ROOT.exists():
        raise FileNotFoundError("Không thấy /kaggle/input. Hãy thêm Dataset bằng Add Input.")
    input_roots = sorted(path for path in INPUT_ROOT.iterdir() if path.is_dir())
    print(f"Các thư mục input gắn vào: {[path.name for path in input_roots]}", flush=True)
    candidates = []
    for input_root in input_roots:
        for train_csv in input_root.rglob("train_subset0.csv"):
            labels_dir = train_csv.parent
            required_csvs = [labels_dir / "labels.csv"] + [
                labels_dir / f"{split}_subset0.csv" for split in ("train", "val", "test")
            ]
            if all(path.is_file() for path in required_csvs):
                if labels_dir.parent.name.lower() == "data" and labels_dir.name.lower() == "labels":
                    dataset_root = labels_dir.parent.parent
                else:
                    dataset_root = labels_dir.parent
                if DATASET_SLUG and DATASET_SLUG.lower() not in str(dataset_root).lower():
                    continue
                candidates.append((dataset_root, labels_dir))
    candidates = sorted(set(candidates))
    if len(candidates) != 1:
        expected = "<Dataset>/data/labels/{labels.csv, train_subset0.csv, val_subset0.csv, test_subset0.csv}"
        raise RuntimeError(
            f"Cần đúng một Dataset chứa CSV fold 0; tìm thấy {candidates}. "
            f"Cấu trúc cần có: {expected}. Điền DATASET_SLUG nếu đã gắn nhiều Dataset."
        )
    return candidates[0]

DATASET_ROOT, LABELS_DIR = find_dataset_and_labels()
print(f"Dataset được chọn: {DATASET_ROOT}", flush=True)
print(f"CSV fold 0 được đọc từ: {LABELS_DIR}", flush=True)
labels_metadata = pd.read_csv(LABELS_DIR / "labels.csv", nrows=0)
if {"Label", "Species"}.issubset(labels_metadata.columns):
    EVAL_LABELS_PATH = LABELS_DIR / "labels.csv"
    LABEL_NAMES = eval_utils.load_names(str(EVAL_LABELS_PATH))
    print("Tên lớp lấy từ labels.csv.", flush=True)
else:
    EVAL_LABELS_PATH = None
    LABEL_NAMES = eval_utils.load_names(None)
    print("labels.csv không có cột Species; dùng tên lớp chuẩn của eval.py.", flush=True)
train_df, val_df, test_df = dataset.load_split(LABELS_DIR, fold=0)
required_images = set(train_df.Filename.astype(str)) | set(val_df.Filename.astype(str)) | set(test_df.Filename.astype(str))

image_candidates = []
for candidate in (LABELS_DIR.parent / "images", DATASET_ROOT / "data" / "images", DATASET_ROOT / "images"):
    if candidate.is_dir() and candidate not in image_candidates:
        image_candidates.append(candidate)
IMAGE_SOURCE_DIR = next((path for path in image_candidates
                         if all((path / name).is_file() for name in required_images)), None)

if IMAGE_SOURCE_DIR is not None:
    IMAGE_DIR = IMAGE_SOURCE_DIR
    print(f"Ảnh được đọc trực tiếp từ Kaggle Dataset: {IMAGE_DIR}", flush=True)
else:
    EXTRACT_DIR = None
    archives = sorted(DATASET_ROOT.rglob("images.zip"))
    if len(archives) > 1:
        raise RuntimeError(f"Có nhiều images.zip trong Dataset {DATASET_ROOT}: {archives}")
    if archives:
        archive = archives[0]
        checksum = hashlib.md5()
        with archive.open("rb") as stream:
            for chunk in iter(lambda: stream.read(1024 * 1024), b""):
                checksum.update(chunk)
        digest = checksum.hexdigest()
        print(f"images.zip: {archive} | MD5={digest}", flush=True)
        EXTRACT_DIR = WORK / "deepweeds" / "extracted" / DATASET_ROOT.name / digest
        EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
        if digest == "b7b30f96d466fba86016aa5a26606e0f":
            print("MD5 khớp file DeepWeeds gốc.", flush=True)
        else:
            print("MD5 khác bản lưu trong README; vẫn kiểm tra đủ ảnh theo CSV fold 0.", flush=True)
        with zipfile.ZipFile(archive) as zf:
            root = EXTRACT_DIR.resolve()
            for member in zf.infolist():
                target = (EXTRACT_DIR / member.filename).resolve()
                if target != root and root not in target.parents:
                    raise ValueError(f"Đường dẫn không an toàn trong zip: {member.filename}")
            zf.extractall(EXTRACT_DIR)

    image_sources = {}
    search_roots = [DATASET_ROOT]
    if EXTRACT_DIR is not None:
        search_roots.insert(0, EXTRACT_DIR)
    for search_root in search_roots:
        for path in search_root.rglob("*"):
            if path.is_file() and path.name in required_images:
                image_sources.setdefault(path.name, []).append(path)
    missing = sorted(required_images - set(image_sources))
    if missing:
        raise FileNotFoundError(
            f"Thiếu {len(missing)} ảnh trong Dataset {DATASET_ROOT}; ví dụ: {missing[:8]}. "
            "Kiểm tra data/images hoặc images.zip."
        )
    for name, paths in image_sources.items():
        if len(paths) > 1:
            extracted = [path for path in paths if EXTRACT_DIR is not None and EXTRACT_DIR in path.parents]
            if len(extracted) == 1:
                image_sources[name] = extracted
            else:
                raise RuntimeError(f"Tên ảnh không duy nhất trong Dataset: {name}: {paths[:3]}")

    IMAGE_DIR = WORK / "deepweeds" / "images"
    IMAGE_DIR.mkdir(parents=True, exist_ok=True)
    linked = 0
    for name in tqdm(sorted(required_images), desc="Chuẩn bị ảnh fold 0", unit="ảnh"):
        destination = IMAGE_DIR / name
        source = image_sources[name][0]
        if destination.exists() or destination.is_symlink():
            continue
        try:
            destination.symlink_to(source)
        except OSError:
            shutil.copy2(source, destination)
        linked += 1
    print(f"Ảnh đã liên kết/copy từ Dataset {DATASET_ROOT} vào {IMAGE_DIR}: {linked}", flush=True)
print(f"Sử dụng {len(required_images):,} ảnh thuộc fold 0.", flush=True)


## EDA và kiểm tra split

Kiểm tra fold 0, tên ảnh giao nhau, hợp 17.509 ảnh và sự tồn tại của toàn bộ ảnh. Notebook in số lượng theo lớp và hiển thị ít nhất ba ảnh train mỗi lớp.

In [ ]:
from IPython.display import display
import matplotlib.pyplot as plt
from PIL import Image

split_info = dataset.check_split(train_df, val_df, test_df, IMAGE_DIR, verbose=True)
class_names = LABEL_NAMES
count_table = pd.DataFrame({
    split: frame.Label.astype(int).value_counts().reindex(range(9), fill_value=0).to_numpy()
    for split, frame in (("train", train_df), ("val", val_df), ("test", test_df))
}, index=class_names)
display(count_table.rename_axis("Lớp").reset_index())
print("Tỉ lệ train/val/test:", split_info["ratios"], flush=True)

ax = count_table.plot(kind="bar", figsize=(15, 5), title="Fold 0 — phân bố lớp theo split")
ax.set_ylabel("Số ảnh")
ax.set_xlabel("Lớp")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()
plt.close("all")

sample_rows = train_df.groupby("Label", sort=True).head(3).sort_values("Label")
fig, axes = plt.subplots(9, 3, figsize=(10, 23))
for row_index, (_, row) in enumerate(sample_rows.reset_index(drop=True).iterrows()):
    ax = axes[row_index // 3, row_index % 3]
    with Image.open(IMAGE_DIR / str(row.Filename)) as image:
        ax.imshow(image.convert("RGB"))
    ax.set_title(f"{class_names[int(row.Label)]} | {row.Filename}", fontsize=8)
    ax.axis("off")
fig.suptitle("Ba ảnh train mẫu cho mỗi lớp", y=1.0)
plt.tight_layout()
plt.show()
plt.close("all")

paper_counts = np.array([1125,1064,1031,1022,1062,1009,1074,1016,9106])
comparison = pd.DataFrame({"class":class_names,"actual":count_table.sum(axis=1).to_numpy(),
                           "paper_Table1":paper_counts})
comparison["difference"] = comparison.actual - comparison.paper_Table1
display(comparison)
print("Largest/smallest class ratio:", comparison.actual.max()/comparison.actual.min(), flush=True)
count_table.to_csv(OUTPUT_ROOT / "split_counts.csv")
atomic_json(OUTPUT_ROOT / "split_check.json", split_info)
with Image.open(IMAGE_DIR / str(train_df.iloc[0].Filename)) as sample:
    print(f"Sample image size={sample.size}, mode={sample.mode}", flush=True)
fig, ax = plt.subplots(figsize=(12,4))
count_table.plot.bar(ax=ax)
fig.tight_layout()
fig.savefig(OUTPUT_ROOT / "class_distribution.png",dpi=150)
plt.close(fig)


## Smoke test pipeline

Đo loss ban đầu trên 9 lớp, sau đó thử overfit một batch nhỏ và hiển thị ảnh sau augmentation. Đây là kiểm tra trước khi chạy các thí nghiệm dài; nếu loss không giảm rõ, dừng để xem log và kiểm tra GPU/data.

In [ ]:
pipeline_path = OUTPUT_ROOT / "pipeline_check.json"
pipeline_saved = json.loads(pipeline_path.read_text()) if pipeline_path.exists() else {}
if STAGE == "prepare" or pipeline_saved.get("source") != EXPECTED_SOURCE_SHA256:
    import math
    import torch
    from torch.utils.data import DataLoader

    small_rows = train_df.groupby("Label", sort=True).head(1).head(4).reset_index(drop=True)
    small_ds = dataset.DeepWeedsDataset(small_rows, IMAGE_DIR, dataset.build_transforms(False, 128))
    small_x, small_y, small_names = next(iter(DataLoader(small_ds, batch_size=len(small_ds), shuffle=False)))
    smoke_model = model_utils.build_model("resnet18", pretrained=True, num_classes=9).to(DEVICE)
    smoke_model.train()
    smoke_loss = torch.nn.CrossEntropyLoss()
    with torch.inference_mode():
        initial_loss = float(smoke_loss(smoke_model(small_x.to(DEVICE)), small_y.to(DEVICE)).item())
    print(f"Initial CE={initial_loss:.4f}; reference ln(9)={math.log(9):.4f}; batch={small_names}", flush=True)
    smoke_optimizer = torch.optim.AdamW(smoke_model.parameters(), lr=1e-3)
    steps = 120
    for step in tqdm(range(steps), desc="Overfit 4 ảnh", unit="step"):
        smoke_model.train()
        smoke_optimizer.zero_grad(set_to_none=True)
        loss = smoke_loss(smoke_model(small_x.to(DEVICE)), small_y.to(DEVICE))
        loss.backward()
        smoke_optimizer.step()
        if (step + 1) % 10 == 0 or step + 1 == steps:
            print(f"overfit step={step + 1}/{steps} loss={loss.item():.4f}", flush=True)
    smoke_model.eval()
    with torch.inference_mode():
        final_smoke_loss = float(smoke_loss(smoke_model(small_x.to(DEVICE)), small_y.to(DEVICE)).item())
    print(f"Kết quả overfit: {initial_loss:.4f} -> {final_smoke_loss:.4f}", flush=True)
    if final_smoke_loss >= initial_loss:
        raise RuntimeError("Loss không giảm trong smoke test; kiểm tra nhãn, GPU và forward/backward trước khi chạy experiment.")
    if final_smoke_loss >= 0.1:
        raise RuntimeError("Overfit loss chưa <0.1; kiểm tra pipeline trước khi bật screen.")
    del smoke_model, smoke_optimizer
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

    aug_ds = dataset.DeepWeedsDataset(small_rows, IMAGE_DIR, dataset.build_transforms(True, 224, "color"))
    fig, axes = plt.subplots(1, len(aug_ds), figsize=(14, 4))
    mean = torch.tensor(dataset.IMAGENET_MEAN).view(3, 1, 1)
    std = torch.tensor(dataset.IMAGENET_STD).view(3, 1, 1)
    for ax, (image, label, name) in zip(axes, (aug_ds[i] for i in range(len(aug_ds)))):
        shown = (image * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()
        ax.imshow(shown)
        ax.set_title(f"{class_names[label]}\n{name}", fontsize=8)
        ax.axis("off")
    plt.tight_layout()
    plt.show()

    plt.close("all")
    atomic_json(OUTPUT_ROOT / "pipeline_check.json", {"source":EXPECTED_SOURCE_SHA256,
        "initial_ce":initial_loss,"final_ce":final_smoke_loss,"steps":steps,"passed":True})
    del small_x, small_y, small_ds, aug_ds, loss
    import gc
    gc.collect()
    torch.cuda.empty_cache()
    print("Pipeline đã qua. Nếu STAGE='prepare', dừng duyệt EDA/overfit rồi đổi sang 'screen'.",flush=True)
else:
    print("Reuse pipeline check đã đạt cho đúng source; EDA/split đã kiểm tra lại.", flush=True)


## Helpers, log và checkpoint

Mọi lần chạy ghi `config`, checkpoint tốt nhất theo macro-F1 validation, checkpoint resume theo epoch, history và đường cong trong `/kaggle/working/day2_output`. Có thể chạy lại cell sau khi session bị ngắt; các epoch đã lưu sẽ được resume.

In [ ]:
import time
from dataclasses import asdict

RECORD_PATH = OUTPUT_ROOT / "experiment_records.json"
INFERENCE_PATH = OUTPUT_ROOT / "inference_records.json"
LATENCY_PATH = OUTPUT_ROOT / "latency_records.json"
RUN_RECORDS = json.loads(RECORD_PATH.read_text(encoding="utf-8")) if RECORD_PATH.exists() else {}
INFERENCE_RECORDS = json.loads(INFERENCE_PATH.read_text(encoding="utf-8")) if INFERENCE_PATH.exists() else {}
LATENCY_RECORDS = json.loads(LATENCY_PATH.read_text(encoding="utf-8")) if LATENCY_PATH.exists() else []

BASE_RECIPE = {
    "backbone": "resnet50", "init": "finetune", "drop_rate": 0.0, "img_size": 224,
    "aug": "basic", "sampler": None, "mix": None, "mix_alpha": 1.0,
    "loss": "ce", "label_smoothing": 0.0, "focal_gamma": 2.0,
    "class_weight_beta": None, "epochs": EPOCHS, "batch_size": BATCH_SIZE,
    "lr_backbone": 1e-4, "lr_head": 1e-3, "weight_decay": 0.05,
    "warmup_epochs": 1.0, "ema_decay": None, "amp": True, "num_workers": NUM_WORKERS,
}

def make_config(exp_id, seed, recipe, *, test=False, method="I00", aggregate="prob", temperature=False):
    training_recipe = {key: value for key, value in recipe.items() if key != "data_parallel"}
    return train.Config(
        exp_id=exp_id, seed=int(seed), fold=0, **training_recipe,
        data_parallel=USE_DATA_PARALLEL, convnext_data_parallel=B03_USE_DATA_PARALLEL,
        channels_last=True, cpu_threads=1,
        validate_data=False, images_dir=str(IMAGE_DIR), labels_dir=str(LABELS_DIR),
        out_dir=str(RUNS_DIR), pred_dir=str(PRED_DIR), curves_dir=str(CURVES_DIR),
        resume=True, save_test_predictions=bool(test), inference_method=method,
        aggregate_space=aggregate, temperature_scale=bool(temperature),
    )

def save_json(path, value):
    atomic_json(path, value)

def record_run(result, group, recipe, *, method="I00", aggregate="prob", temperature=False):
    test_metrics = result.get("test_metrics") or {}
    clean_test = {key: float(test_metrics[key]) for key in ("top1", "macro_f1", "balanced_acc", "ece", "nll")
                  if key in test_metrics}
    row = {key: result.get(key) for key in (
        "exp_id", "seed", "backbone", "pretrained_tag", "best_epoch", "checkpoint_macro_f1_val",
        "macro_f1_val", "top1_val", "val_loss", "epochs_completed", "train_seconds_per_epoch",
        "params_m", "gmacs", "checkpoint", "curve", "val_predictions", "temperature",
        "gmac_method", "data_parallel_active", "latency_p50_batch1_ms", "latency_p95_batch1_ms", "latency_gpu")}
    row.update({"group": group, "recipe": recipe, "method": method, "aggregate": aggregate,
                "temperature_scale": bool(temperature), "test_metrics": clean_test})
    RUN_RECORDS[f"{row['exp_id']}_seed{row['seed']}"] = row
    save_json(RECORD_PATH, RUN_RECORDS)
    return row

def execute_training(exp_id, seed, group, recipe, *, test=False, method="I00", aggregate="prob", temperature=False):
    cfg = make_config(exp_id, seed, recipe, test=test, method=method,
                      aggregate=aggregate, temperature=temperature)
    print(f"\n▶ Start {exp_id} | seed={seed} | backbone={recipe['backbone']} | test={test}", flush=True)
    result = runner.launch(cfg)
    return record_run(result, group, recipe, method=method, aggregate=aggregate, temperature=temperature)

def load_best_model(exp_id, seed, recipe):
    model = model_utils.build_model(recipe["backbone"], pretrained=False, num_classes=9,
                                   drop_rate=recipe.get("drop_rate", 0.0)).cpu()
    checkpoint_path = RUNS_DIR / exp_id / f"seed{seed}" / "checkpoint_best.pt"
    checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
    model.load_state_dict(checkpoint["model_state"])
    return model.to(DEVICE).eval()

def validation_metrics(names, labels, probs):
    metrics = eval_utils.compute_metrics(np.asarray(labels), np.asarray(probs).argmax(1), np.asarray(probs))
    return {"n": int(metrics["n"]), "macro_f1_val": float(metrics["macro_f1"]),
            "top1_val": float(metrics["top1"]), "ece_val": float(metrics["ece"]),
            "balanced_acc_val": float(metrics["balanced_acc"])}

print(f"Output root: {OUTPUT_ROOT}", flush=True)


## Backbone benchmark — hai nhóm độc lập

Chạy B01–B02 rồi B03–B05 ở hai cell riêng. Mỗi nhóm dùng cùng công thức nền, fold 0 và seed 0. Cell tự bỏ qua thí nghiệm đã có đầy đủ record, checkpoint, biểu đồ, dự đoán validation và latency. Khi đổi số worker để giảm tải CPU, các kết quả B01–B02 đã hoàn thành vẫn có thể được dùng lại. Nếu chuyển sang Kaggle session mới, hãy khôi phục `/kaggle/working/day2_output` trước khi bỏ qua nhóm đầu.

In [ ]:
import gc
def run_backbone_group(backbones):
    if "BASE_RECIPE" not in globals():
        raise RuntimeError("Chạy từ đầu notebook bằng Run All để khởi tạo setup/helpers; checkpoint đã xong sẽ được reuse.")
    for exp_id, backbone_name in backbones:
        if BACKBONE_FILTER and exp_id not in BACKBONE_FILTER:
            continue
        recipe = {**BASE_RECIPE, "backbone":backbone_name}
        record = execute_training(exp_id,0,"Backbones",recipe)
        RUN_RECORDS[f"{exp_id}_seed0"] = record
        save_json(RECORD_PATH,RUN_RECORDS)
        gc.collect()


### Nhóm 1 — B01 ResNet-50 / B02 ResNeXt-50
Mỗi model chạy trong worker riêng, DataParallel khi thấy hai GPU.
Để bỏ qua nhóm này ở lần chạy sau: `RUN_BACKBONES_B01_B02=False`; vẫn cần giữ checkpoint và records.


In [ ]:
BACKBONES_B01_B02 = [("B01", "resnet50"), ("B02", "resnext50_32x4d")]
if RUN_BACKBONES and RUN_BACKBONES_B01_B02:
    run_backbone_group(BACKBONES_B01_B02)
else:
    print("Bỏ qua nhóm backbone B01–B02.", flush=True)


### Nhóm 2 — B03 ConvNeXt / B04 DeiT / B05 EfficientNet
ConvNeXt mặc định một T4, channels-last + AMP, worker=0, cache cuDNN có giới hạn.
Mặc định này cũng áp dụng khi ConvNeXt đi vào T/F. Chưa xác nhận hết lỗi OOM trên T4.
B04/B05 dùng DataParallel khi có hai GPU. Không đổi batch/epoch giữa các backbone.


In [ ]:
BACKBONES_B03_B05 = [
    ("B03", "convnext_tiny"),
    ("B04", "deit_small_patch16_224"),
    ("B05", "efficientnet_b0"),
]
if RUN_BACKBONES and RUN_BACKBONES_B03_B05:
    run_backbone_group(BACKBONES_B03_B05)
else:
    print("Bỏ qua nhóm backbone B03–B05.", flush=True)


In [ ]:
backbone_rows = sorted(
    [row for row in RUN_RECORDS.values() if row.get("group") == "Backbones" and row.get("seed") == 0],
    key=lambda row: row["macro_f1_val"], reverse=True,
)
if backbone_rows:
    print(pd.DataFrame([{key: row.get(key) for key in (
        "exp_id", "backbone", "pretrained_tag", "params_m", "gmacs", "macro_f1_val",
        "top1_val", "train_seconds_per_epoch", "latency_p50_batch1_ms")}
        for row in backbone_rows]).to_string(index=False), flush=True)
    print(f"Chọn backbone tiếp theo bằng macro-F1 validation: {backbone_rows[0]['backbone']}", flush=True)


## Ablation công thức train — 3 trục và một tổ hợp

So sánh T00 baseline với frozen backbone (khởi tạo), color augmentation và label smoothing/focal (loss). T05 ghép những giá trị thắng trên validation. Chọn model và trục chỉ bằng macro-F1 val; tập test chưa được đọc để chọn.

In [ ]:
def best_backbone_record():
    candidates = [r for r in RUN_RECORDS.values() if r["group"]=="Backbones" and r["seed"]==0]
    if {r["exp_id"] for r in candidates} != {"B01","B02","B03","B04","B05"}:
        raise RuntimeError("Cần đủ năm backbone trước khi chọn backbone cho ablation.")
    return max(candidates,key=lambda r:(r["macro_f1_val"],-r.get("latency_p95_batch1_ms",1e9)))

def best_training_record():
    candidates = [r for r in RUN_RECORDS.values() if r["exp_id"].startswith("T") and r["seed"]==0]
    if len(candidates)<11:
        raise RuntimeError("Cần hoàn tất T00–T10 trước khi chọn công thức.")
    return max(candidates,key=lambda r:r["macro_f1_val"])

ablation_rows=[]
if RUN_ABLATIONS:
    baseline_recipe={**BASE_RECIPE,"backbone":best_backbone_record()["backbone"]}
    variants=[
        ("T00","baseline",{}),
        ("T01","A: init=scratch",{"init":"scratch"}),
        ("T02","A: init=frozen",{"init":"frozen"}),
        ("T03","B: color jitter",{"aug":"color"}),
        ("T04","B: Mixup",{"mix":"mixup"}),
        ("T05","B: CutMix",{"mix":"cutmix"}),
        ("T06","C: label smoothing 0.1",{"loss":"ls","label_smoothing":0.1}),
        ("T07","C: focal gamma 2",{"loss":"focal"}),
        ("T08","C: class-weighted CE",{"loss":"ce_weighted"}),
        ("T09","F: EMA 0.999",{"ema_decay":0.999}),
    ]
    for exp_id,description,delta in variants:
        recipe={**baseline_recipe,**delta}
        row=execute_training(exp_id,0,"Training",recipe)
        row["description"]=description
        RUN_RECORDS[f"{exp_id}_seed0"]=row
        save_json(RECORD_PATH,RUN_RECORDS)
    # Always test an actual combination (augmentation + loss), even if neither wins alone.
    aug_id=max(("T03","T04","T05"),key=lambda k:RUN_RECORDS[k+"_seed0"]["macro_f1_val"])
    loss_id=max(("T06","T07","T08"),key=lambda k:RUN_RECORDS[k+"_seed0"]["macro_f1_val"])
    combo=dict(baseline_recipe)
    for key in ("aug","mix","mix_alpha"):
        combo[key]=RUN_RECORDS[aug_id+"_seed0"]["recipe"][key]
    for key in ("loss","label_smoothing","focal_gamma","class_weight_beta"):
        combo[key]=RUN_RECORDS[loss_id+"_seed0"]["recipe"][key]
    row=execute_training("T10",0,"Training",combo)
    row["description"]=f"B+C combination: {aug_id}+{loss_id} (chosen on val; may not improve baseline)"
    RUN_RECORDS["T10_seed0"]=row
    save_json(RECORD_PATH,RUN_RECORDS)
ablation_rows=[r for r in RUN_RECORDS.values() if r["exp_id"].startswith("T") and r["seed"]==0]
if ablation_rows:
    print(pd.DataFrame([{k:r.get(k) for k in ("exp_id","description","macro_f1_val","top1_val")}
                        for r in ablation_rows]).to_string(index=False),flush=True)


## Inference và latency — chọn chỉ bằng validation

So sánh I00 một view, I01 lật ngang, I02 năm crop, I03 gộp logit của hai view và I08 AMP. I07 temperature scaling fit trên validation từ phương pháp tốt nhất; nó không thay đổi nhãn dự đoán. Latency có 10 warmup, 50 lần đo, synchronize GPU và báo p50/p95/p99 ở batch 1 và 32. Phần đo model forward loại trừ decode/tiền xử lý ảnh.

In [ ]:
inference_rows = []
if RUN_INFERENCE:
    selected_training = best_training_record()
    source_exp = selected_training["exp_id"]
    recipe = selected_training["recipe"]
    model = load_best_model(source_exp, 0, recipe)
    _, val_df, _ = dataset.load_split(LABELS_DIR, fold=0)
    val_loader = dataset.make_loader(val_df, IMAGE_DIR, dataset.build_transforms(False, recipe["img_size"]),
                                     BATCH_SIZE, train=False, num_workers=NUM_WORKERS, seed=0)
    method_specs = [("I00", "prob", 1, "fp32"), ("I01", "prob", 2, "fp32"),
                    ("I02", "prob", 5, "fp32"), ("I03", "logit", 2, "fp32"),
                    ("I08", "prob", 1, "amp")]
    probs_by_method = {}
    base_rows = []
    for method, aggregate, views, dtype in tqdm(method_specs, desc="Validation inference methods", unit="method"):
        cfg = make_config(method, 0, recipe, method=method, aggregate=aggregate)
        names, labels, probs = train._predict_method(model, val_loader, DEVICE, cfg)
        probs_by_method[method] = probs
        pred_path = PRED_DIR / f"{method}_seed0_val.csv"
        eval_utils.save_predictions(pred_path, names, labels, probs)
        metrics = validation_metrics(names, labels, probs)
        time1 = benchmark.method_latency(model,method,aggregate,batch_size=1,
                                         img_size=recipe["img_size"],device=str(DEVICE))
        time32 = benchmark.method_latency(model,method,aggregate,batch_size=32,
                                          img_size=recipe["img_size"],device=str(DEVICE))
        row = {"exp_id": method, "method": method, "base_method": method, "source_exp": source_exp,
               "description": {"I00": "single view", "I01": "horizontal flip, mean probabilities",
                               "I02": "five crops, mean probabilities", "I03": "horizontal flip, mean logits",
                               "I08": "single view AMP"}[method],
               "checkpoint": str(RUNS_DIR / source_exp / "seed0" / "checkpoint_best.pt"),
               "views": views, "aggregate": aggregate, "dtype": dtype, **metrics,
               "p50_batch1_ms": time1["p50"], "p95_batch1_ms": time1["p95"],
               "p99_batch1_ms": time1["p99"], "images_per_s_batch1": time1["images_per_s"],
               "p50_batch32_ms": time32["p50"], "p95_batch32_ms": time32["p95"],
               "p99_batch32_ms": time32["p99"], "images_per_s_batch32": time32["images_per_s"],
               "gpu": time1["gpu"], "torch": time1["torch"], "bn_fused": False,
               "preprocessing": "base resize/decode/H2D excluded; actual TTA, aggregation included"}
        INFERENCE_RECORDS[method] = row
        base_rows.append(row)
        LATENCY_RECORDS.extend([
            {"configuration": method, "gpu": time1["gpu"], "dtype": dtype, "batch": 1,
             "bn_fused": False, "p50_ms": time1["p50"], "p95_ms": time1["p95"],
             "p99_ms": time1["p99"], "images_per_s": time1["images_per_s"], "torch": time1["torch"]},
            {"configuration": method, "gpu": time32["gpu"], "dtype": dtype, "batch": 32,
             "bn_fused": False, "p50_ms": time32["p50"], "p95_ms": time32["p95"],
             "p99_ms": time32["p99"], "images_per_s": time32["images_per_s"], "torch": time32["torch"]},
        ])
        save_json(INFERENCE_PATH, INFERENCE_RECORDS)
        save_json(LATENCY_PATH, LATENCY_RECORDS)

    best_base = max(base_rows, key=lambda row: (row["macro_f1_val"], -row["ece_val"], -row["p95_batch1_ms"]))
    raw_logits = np.log(np.clip(probs_by_method[best_base["base_method"]], 1e-12, 1.0))
    temperature = inference.fit_temperature(raw_logits, val_df.Label.to_numpy(dtype=int))
    calibrated = inference.apply_temperature(raw_logits, temperature)
    names, labels, _ = train._predict_method(
        model, val_loader, DEVICE,
        make_config(best_base["base_method"], 0, recipe, method=best_base["base_method"],
                    aggregate=best_base["aggregate"]))
    eval_utils.save_predictions(PRED_DIR / "I07_seed0_val.csv", names, labels, calibrated)
    ts_metrics = validation_metrics(names, labels, calibrated)
    ts1 = benchmark.method_latency(model,best_base["base_method"],best_base["aggregate"],temperature,
                                    batch_size=1,img_size=recipe["img_size"],device=str(DEVICE))
    ts32 = benchmark.method_latency(model,best_base["base_method"],best_base["aggregate"],temperature,
                                     batch_size=32,img_size=recipe["img_size"],device=str(DEVICE))
    INFERENCE_RECORDS["I07"] = {
        **best_base, "exp_id": "I07", "method": "I07", "base_method": best_base["base_method"],
        "description": f"temperature scaling on {best_base['base_method']}; T={temperature:.5f}",
        "temperature": temperature, **ts_metrics,
        "p50_batch1_ms":ts1["p50"],"p95_batch1_ms":ts1["p95"],"p99_batch1_ms":ts1["p99"],
        "images_per_s_batch1":ts1["images_per_s"],"images_per_s_batch32":ts32["images_per_s"],
        "p50_batch32_ms":ts32["p50"],"p95_batch32_ms":ts32["p95"],"p99_batch32_ms":ts32["p99"],
        "relative_cost_vs_I00": best_base["p50_batch1_ms"] / max(INFERENCE_RECORDS["I00"]["p50_batch1_ms"], 1e-12),
    }
    save_json(INFERENCE_PATH, INFERENCE_RECORDS)
    for batch,timing in ((1,ts1),(32,ts32)):
        LATENCY_RECORDS.append({"configuration":"I07","gpu":timing["gpu"],"dtype":best_base["dtype"],
            "batch":batch,"bn_fused":False,"p50_ms":timing["p50"],"p95_ms":timing["p95"],
            "p99_ms":timing["p99"],"images_per_s":timing["images_per_s"],"torch":timing["torch"]})
    save_json(LATENCY_PATH,LATENCY_RECORDS)
    inference_rows = list(INFERENCE_RECORDS.values())
    best_inference = max(inference_rows, key=lambda row: (row["macro_f1_val"], -row["ece_val"], -row["p95_batch1_ms"]))
    print(pd.DataFrame([{key: row.get(key) for key in (
        "method", "base_method", "description", "macro_f1_val", "top1_val", "ece_val",
        "p50_batch1_ms", "p95_batch1_ms", "p99_batch1_ms", "images_per_s_batch1")}
        for row in inference_rows]).to_string(index=False), flush=True)
    print(f"Chốt inference theo validation: {best_inference['method']} (base={best_inference['base_method']}, "
          f"temperature={best_inference['method'] == 'I07'})", flush=True)
    del model
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
elif STAGE in {"final","report"}:
    selected_training=best_training_record()
    inference_rows=[r for r in INFERENCE_RECORDS.values() if r.get("source_exp")==selected_training["exp_id"]]
    if len(inference_rows)<6:
        raise RuntimeError("Chưa đủ I00/I01/I02/I03/I08/I07; chạy stage inference trước final.")
    best_inference=max(inference_rows,key=lambda r:(r["macro_f1_val"],-r["ece_val"],-r["p95_batch1_ms"]))
else:
    print("Bỏ qua inference theo STAGE hiện tại.",flush=True)


## Chung kết — test đúng một lần mỗi seed

Đọc lại kết quả validation, sau đó mới bật `RUN_FINAL = True` ở ô cấu hình. Mỗi seed lưu file test một lần; khi chạy lại cell, notebook kiểm tra file có sẵn, đối chiếu toàn bộ tên ảnh/nhãn với CSV gốc rồi bỏ qua inference test. Baseline là T00/I00 và final là recipe tốt nhất đã chọn trên validation.

In [ ]:
def read_or_run_final(exp_id, seed, group, recipe, *, method="I00", aggregate="prob", temperature=False):
    test_csv = PRED_DIR / f"{exp_id}_seed{seed}_test.csv"
    val_csv = PRED_DIR / f"{exp_id}_seed{seed}_val.csv"
    if test_csv.is_file():
        saved = RUN_RECORDS.get(f"{exp_id}_seed{seed}")
        if saved and (saved.get("recipe") != recipe or saved.get("method", "I00") != method
                      or saved.get("aggregate", "prob") != aggregate
                      or bool(saved.get("temperature_scale")) != bool(temperature)):
            raise RuntimeError(f"{test_csv} already exists for a different configuration; refusing to rerun or overwrite test.")
        pred = eval_utils.read_pred(str(test_csv))
        eval_utils.check_against_csv(pred, str(LABELS_DIR / "test_subset0.csv"), what="test")
        metrics = eval_utils.compute_metrics(pred.y_true, pred.y_pred, pred.probs)
        print(f"Đã có {test_csv}; xác minh và không chạy lại test. macro-F1={metrics['macro_f1']:.4f}", flush=True)
        folder = RUNS_DIR / exp_id / f"seed{seed}"
        cfg = make_config(exp_id,seed,recipe,test=True,method=method,aggregate=aggregate,temperature=temperature)
        saved_cfg=json.loads((folder / "config.json").read_text())
        if not train._same_training_config(saved_cfg,cfg):
            raise RuntimeError("Config đã lưu không khớp final; không chạy lại test.")
        for key in ("inference_method","aggregate_space","temperature_scale"):
            if saved_cfg.get(key)!=getattr(cfg,key):
                raise RuntimeError("Inference final khác cấu hình đã dùng cho test.")
        result = RUN_RECORDS.get(f"{exp_id}_seed{seed}")
        if result is None:
            summary=json.loads((folder / "result.json").read_text())
            result=record_run(summary,group,recipe,method=method,aggregate=aggregate,temperature=temperature)
        result = dict(result)
        result["test_metrics"] = {key: float(metrics[key]) for key in ("top1", "macro_f1", "balanced_acc", "ece", "nll")}
        if temperature and not (PRED_DIR / f"{exp_id}_uncal_seed{seed}_test.csv").is_file():
            print("Cảnh báo: thiếu file uncal đã lưu; không đánh giá lại test để tạo file này.", flush=True)
        RUN_RECORDS[f"{exp_id}_seed{seed}"] = result
        save_json(RECORD_PATH, RUN_RECORDS)
        return result
    result = execute_training(exp_id, seed, group, recipe, test=True, method=method,
                              aggregate=aggregate, temperature=temperature)
    return result

if RUN_FINAL:
    if "best_inference" not in globals() or not best_inference:
        raise RuntimeError("Chạy cell inference và kiểm tra kết quả validation trước khi bật chung kết.")
    if len(FINAL_SEEDS) < 3 or len(set(FINAL_SEEDS)) != len(FINAL_SEEDS):
        raise ValueError("RUBRIC yêu cầu ít nhất 3 seed cuối cùng khác nhau.")
    best_train = best_training_record()
    final_recipe = best_train["recipe"]
    final_method = best_inference["base_method"]
    final_temperature = best_inference["method"] == "I07"
    print("Cấu hình final đã chốt trên validation:", {
        "recipe": final_recipe, "method": final_method, "temperature_scale": final_temperature,
        "validation_macro_f1": best_inference["macro_f1_val"]}, flush=True)

    if not SELECTION_REASON.strip():
        raise ValueError("Điền SELECTION_REASON sau khi duyệt validation/latency trước khi mở test.")
    selection={"recipe":final_recipe,"method":final_method,"aggregate":best_inference["aggregate"],
               "temperature":final_temperature,"baseline":RUN_RECORDS["T00_seed0"]["recipe"],
               "seeds":FINAL_SEEDS,"reason":SELECTION_REASON}
    selection_path=OUTPUT_ROOT / "final_selection.json"
    if selection_path.exists() and json.loads(selection_path.read_text())!=selection:
        raise RuntimeError("Cấu hình final đã khóa; không thay đổi sau khi mở test.")
    save_json(selection_path,selection)
    for seed in tqdm(FINAL_SEEDS, desc="Baseline T00/I00 — final test", unit="seed"):
        read_or_run_final("T00", seed, "Final baseline T00", RUN_RECORDS["T00_seed0"]["recipe"], method="I00")
    for seed in tqdm(FINAL_SEEDS, desc="Final recipe — final test", unit="seed"):
        read_or_run_final("F01", seed, "Final F01", final_recipe, method=final_method,
                          aggregate=best_inference.get("aggregate", "prob"), temperature=final_temperature)
else:
    print("STAGE hiện tại không thực hiện lượt test mới.", flush=True)


## Đánh giá, workbook và tệp tải xuống

`eval.py score` đối chiếu mọi dự đoán test với CSV fold 0. `eval.py grade` tạo tự chấm phần I khi có đủ F01/T00, file uncal, validation và latency. Workbook có đủ bảy sheet theo GUIDE. Phần nhận xét/report sẽ được viết sau khi có notebook và kết quả chạy thật.

In [ ]:
def group_paths(exp_id, split):
    return sorted(PRED_DIR.glob(f"{exp_id}_seed*_{split}.csv"))

def evaluate_saved_group(exp_id, split, reference):
    files = group_paths(exp_id, split)
    if not files:
        return None
    return eval_utils.load_group([str(path) for path in files], str(reference), ref_what=split)

def rows_for_group(exp_id, split, reference):
    group = evaluate_saved_group(exp_id, split, reference)
    if group is None:
        return None
    return group

def workbook_dataframes():
    backbones = [row for row in RUN_RECORDS.values() if row["group"] == "Backbones"]
    trainings = [row for row in RUN_RECORDS.values() if row["exp_id"].startswith("T") and row["seed"] == 0]
    back_df = pd.DataFrame([{
        "exp_id": row["exp_id"], "backbone": row["backbone"], "pretrained tag": row["pretrained_tag"],
        "params (M)": row["params_m"], "GMAC (estimated)": row["gmacs"],
        "resolution": row["recipe"]["img_size"], "epochs": row["recipe"]["epochs"], "seed": row["seed"],
        "train DataParallel": row.get("data_parallel_active"),
        "macro-F1 val": row["macro_f1_val"], "top-1 val": row["top1_val"],
        "train sec/epoch": row["train_seconds_per_epoch"],
        "latency p50 batch1 (ms)": row.get("latency_p50_batch1_ms"),
        "latency GPU": row.get("latency_gpu"), "notes": row.get("gmac_method", "")}
        for row in backbones])
    train_df_out = pd.DataFrame([{
        "exp_id": row["exp_id"], "backbone": row["backbone"], "axis": row.get("description", "baseline/combination"),
        "changed from T00": row.get("description", "baseline"), "seed": row["seed"],
        "macro-F1 val": row["macro_f1_val"], "top-1 val": row["top1_val"],
        "delta vs T00": row["macro_f1_val"] - RUN_RECORDS["T00_seed0"]["macro_f1_val"] if "T00_seed0" in RUN_RECORDS else np.nan,
        "train sec/epoch": row["train_seconds_per_epoch"], "init": row["recipe"]["init"],
        "augmentation": row["recipe"]["aug"], "loss": row["recipe"]["loss"],
        "notes": row.get("description", "")}
        for row in trainings])
    inference_rows_local = list(INFERENCE_RECORDS.values())
    i00 = next((row for row in inference_rows_local if row["method"] == "I00"), None)
    inference_df = pd.DataFrame([{
        "exp_id": row["method"], "method": row["description"], "checkpoint": row["checkpoint"],
        "K views": row["views"], "macro-F1 val": row["macro_f1_val"], "top-1 val": row["top1_val"],
        "ECE val": row["ece_val"], "p50 batch1 (ms)": row["p50_batch1_ms"],
        "p95 batch1 (ms)": row["p95_batch1_ms"], "p99 batch1 (ms)": row["p99_batch1_ms"],
        "throughput batch1 (img/s)": row["images_per_s_batch1"],
        "relative cost vs I00": row["p50_batch1_ms"] / max(i00["p50_batch1_ms"], 1e-12) if i00 else np.nan,
        "notes": row["preprocessing"]}
        for row in inference_rows_local])

    final_rows, per_class_rows = [], []
    for exp_id in ("T00", "F01"):
        test_group = rows_for_group(exp_id, "test", LABELS_DIR / "test_subset0.csv")
        val_group = rows_for_group(exp_id, "val", LABELS_DIR / "val_subset0.csv")
        if test_group is None:
            continue
        for pred, metric in zip(test_group.preds, test_group.metrics):
            val_metric = None
            if val_group:
                val_metric = next((m for p, m in zip(val_group.preds, val_group.metrics) if p.seed == pred.seed), None)
            final_rows.append({"exp_id": exp_id, "configuration": json.dumps({"recipe":RUN_RECORDS.get(f"{exp_id}_seed{pred.seed}",{}).get("recipe"),
                               "method":RUN_RECORDS.get(f"{exp_id}_seed{pred.seed}",{}).get("method"),
                               "temperature_scale":RUN_RECORDS.get(f"{exp_id}_seed{pred.seed}",{}).get("temperature_scale")}),
                               "seed": pred.seed, "macro-F1 val": val_metric["macro_f1"] if val_metric else np.nan,
                               "macro-F1 test": metric["macro_f1"], "top-1 test": metric["top1"], "ECE test": metric["ece"]})
        final_rows.append({"exp_id": exp_id, "configuration": "mean ± sample std (ddof=1)", "seed": "summary",
                           "macro-F1 val": (val_group.summary["macro_f1"][0] if val_group else np.nan),
                           "macro-F1 val std": (val_group.summary["macro_f1"][1] if val_group else np.nan),
                           "macro-F1 test": test_group.summary["macro_f1"][0],
                           "macro-F1 test std": test_group.summary["macro_f1"][1],
                           "top-1 test": test_group.summary["top1"][0], "top-1 test std": test_group.summary["top1"][1],
                           "ECE test": test_group.summary["ece"][0], "ECE test std": test_group.summary["ece"][1]})
        names = class_names
        prec_mean, prec_std = test_group.summary["precision"]
        rec_mean, rec_std = test_group.summary["recall"]
        f1_mean, f1_std = test_group.summary["f1"]
        support = test_group.metrics[0]["support"].astype(int)
        for i, name in enumerate(names):
            per_class_rows.append({"exp_id": exp_id, "class": name, "test support": support[i],
                                   "precision mean": prec_mean[i], "precision std": prec_std[i],
                                   "recall mean": rec_mean[i], "recall std": rec_std[i],
                                   "F1 mean": f1_mean[i], "F1 std": f1_std[i]})
    latency_df = pd.DataFrame(LATENCY_RECORDS)
    if not latency_df.empty:
        latency_df = latency_df.drop_duplicates(subset=["configuration", "batch"], keep="last")
    else:
        latency_df = pd.DataFrame(columns=["configuration", "gpu", "dtype", "batch", "bn_fused",
                                          "p50_ms", "p95_ms", "p99_ms", "images_per_s", "torch"])
    comparison = []
    for row in list(backbones) + list(trainings) + inference_rows_local:
        macro = row.get("macro_f1_val")
        if macro is not None:
            comparison.append({"group": row.get("group", "Inference"), "exp_id": row.get("exp_id"),
                               "configuration": row.get("description", row.get("backbone", row.get("method"))),
                               "macro-F1 val": macro, "top-1 val": row.get("top1_val"),
                               "latency p95 batch1 (ms)": row.get("p95_batch1_ms", row.get("latency_p95_batch1_ms")),
                               "params (M)": row.get("params_m"), "GMAC": row.get("gmacs")})
    summary_df = pd.DataFrame(comparison).sort_values("macro-F1 val", ascending=False).head(10) if comparison else pd.DataFrame()
    return {"Backbones": back_df, "Training": train_df_out, "Inference": inference_df,
            "Final": pd.DataFrame(final_rows), "PerClass": pd.DataFrame(per_class_rows),
            "Latency": latency_df, "Summary": summary_df}

def write_results_workbook():
    output = OUTPUT_ROOT / "results.xlsx"
    sheets = workbook_dataframes()
    with pd.ExcelWriter(output, engine="openpyxl") as writer:
        for sheet, frame in sheets.items():
            frame.to_excel(writer, sheet_name=sheet, index=False)
            ws = writer.sheets[sheet]
            ws.freeze_panes = "A2"
            from openpyxl.styles import Font, PatternFill
            from openpyxl.utils import get_column_letter
            for cell in ws[1]:
                cell.font=Font(bold=True,color="FFFFFF")
                cell.fill=PatternFill("solid",fgColor="17324D")
            for col in ws.columns:
                ws.column_dimensions[get_column_letter(col[0].column)].width=min(50,max(15,len(str(col[0].value))+2))
            for row in ws.iter_rows(min_row=2):
                for cell in row:
                    if isinstance(cell.value,float): cell.number_format="0.0000"
            if "macro-F1 val" in frame.columns and not frame.empty:
                best=int(frame["macro-F1 val"].fillna(-1).to_numpy().argmax())+2
                for cell in ws[best]: cell.fill=PatternFill("solid",fgColor="DCFCE7")
            if ws.max_column:
                ws.auto_filter.ref = ws.dimensions
    print(f"Đã ghi workbook: {output}", flush=True)
    return output

if STAGE in {"final","report"}:
    test_files_final = group_paths("F01", "test")
    test_files_base = group_paths("T00", "test")
    if test_files_final and test_files_base:
        for exp_id, files in (("F01", test_files_final), ("T00", test_files_base)):
            print(f"\n### eval.py score — {exp_id}", flush=True)
            command = [sys.executable, "-X", "utf8", str(CODE_DIR / "eval.py"), "score", "--pred", *map(str, files),
                       "--test-csv", str(LABELS_DIR / "test_subset0.csv"),
                       "--tag", exp_id, "--out", str(EVAL_DIR)]
            if EVAL_LABELS_PATH is not None:
                command += ["--labels", str(EVAL_LABELS_PATH)]
            subprocess.run(command, check=True)
        final_val_files = group_paths("F01", "val")
        uncal_files = sorted(PRED_DIR.glob("F01_uncal_seed*_test.csv"))
        best_p95 = best_inference.get("p95_batch1_ms")
        grade_cmd = [sys.executable, "-X", "utf8", str(CODE_DIR / "eval.py"), "grade",
                     "--final", *map(str, test_files_final), "--baseline", *map(str, test_files_base),
                     "--test-csv", str(LABELS_DIR / "test_subset0.csv"),
                     "--latency-p95-ms", str(best_p95),
                     "--latency-method", "proper", "--out", str(EVAL_DIR)]
        if EVAL_LABELS_PATH is not None:
            grade_cmd += ["--labels", str(EVAL_LABELS_PATH)]
        if uncal_files:
            grade_cmd += ["--uncal", *map(str, uncal_files)]
        if final_val_files:
            grade_cmd += ["--final-val", *map(str, final_val_files), "--val-csv", str(LABELS_DIR / "val_subset0.csv")]
        subprocess.run(grade_cmd, check=True)
    else:
        print("Chưa đủ F01/T00 predictions để chạy score/grade.", flush=True)

# Data-backed evidence for the later report; no generated conclusions or invented scores.
if INFERENCE_RECORDS:
    fig,ax=plt.subplots(figsize=(8,5))
    for row in INFERENCE_RECORDS.values():
        ax.scatter(row["p95_batch1_ms"],row["macro_f1_val"])
        ax.annotate(row["method"],(row["p95_batch1_ms"],row["macro_f1_val"]))
    ax.set(xlabel="p95 latency, batch 1 (ms)",ylabel="Macro-F1 validation",title="Validation accuracy / latency")
    fig.tight_layout(); fig.savefig(OUTPUT_ROOT / "inference_tradeoff.png",dpi=150); plt.close(fig)
if STAGE in {"final","report"} and group_paths("F01","test"):
    group=evaluate_saved_group("F01","test",LABELS_DIR / "test_subset0.csv")
    cm=sum(m["confusion"] for m in group.metrics)
    fig,ax=plt.subplots(figsize=(11,9))
    im=ax.imshow(cm,cmap="Blues")
    ax.set(xticks=range(9),yticks=range(9),xticklabels=class_names,yticklabels=class_names,
           xlabel="Predicted",ylabel="True",title="F01 test confusion (sum across seeds)")
    plt.setp(ax.get_xticklabels(),rotation=40,ha="right")
    for y in range(9):
        for x in range(9): ax.text(x,y,str(cm[y,x]),ha="center",va="center",fontsize=8)
    fig.colorbar(im,ax=ax); fig.tight_layout(); fig.savefig(OUTPUT_ROOT / "confusion_F01.png",dpi=150); plt.close(fig)
    pred=group.preds[0]
    mistakes=np.flatnonzero(pred.y_pred!=pred.y_true)
    # Inspect hard-class confusion first, then other errors. These are test diagnostics after locking final.
    hard=[i for i in mistakes if {int(pred.y_true[i]),int(pred.y_pred[i])}=={0,7}]
    selected=(hard+[i for i in mistakes if i not in set(hard)])[:12]
    if selected:
        fig,axes=plt.subplots(3,4,figsize=(14,10))
        for ax in axes.flat: ax.axis("off")
        for ax,i in zip(axes.flat,selected):
            with Image.open(IMAGE_DIR / pred.filenames[i]) as img: ax.imshow(img.convert("RGB"))
            ax.set_title(f"True: {class_names[pred.y_true[i]]}\nPred: {class_names[pred.y_pred[i]]}",fontsize=8)
        fig.tight_layout(); fig.savefig(OUTPUT_ROOT / "errors_F01.png",dpi=150); plt.close(fig)
workbook_path = write_results_workbook()
runtime_file = OUTPUT_ROOT / "runtime.json"
from importlib.metadata import version
package_versions = {name: version(name) for name in
                    ("torch", "torchvision", "timm", "numpy", "pandas", "Pillow", "matplotlib", "tqdm")}
save_json(runtime_file, {"python": sys.version, "torch": torch.__version__,
                         "cuda": torch.version.cuda, "gpu": torch.cuda.get_device_name(DEVICE) if DEVICE.type == "cuda" else "CPU",
                         "gpu_count": GPU_COUNT, "gpu_names": GPU_NAMES,
                         "cpu_threads": torch.get_num_threads(), "data_loader_workers": NUM_WORKERS,
                         "data_parallel_requested": USE_DATA_PARALLEL,
                         "b03_data_parallel": B03_USE_DATA_PARALLEL,
                         "data_parallel_active": DATA_PARALLEL_ACTIVE,
                         "packages": package_versions,
                         "labels_dir": str(LABELS_DIR), "images_dir": str(IMAGE_DIR),
                         "fold": 0, "final_seeds": FINAL_SEEDS})
print("Output files:", flush=True)
for path in sorted(OUTPUT_ROOT.rglob("*")):
    if path.is_file():
        print(f" - {path} ({path.stat().st_size:,} bytes)", flush=True)
if PACKAGE_RESUME_OUTPUT:
    resume_zip = shutil.make_archive(str(WORK / "day2_output"), "zip", root_dir=WORK,
                                     base_dir=OUTPUT_ROOT.name)
    print(f"Resume archive (may be large): {resume_zip}", flush=True)

from zipfile import ZipFile, ZIP_DEFLATED
submission_zip=WORK / "day2_submission.zip"
with ZipFile(submission_zip,"w",ZIP_DEFLATED) as archive:
    for path in OUTPUT_ROOT.rglob("*"):
        if path.is_file() and path.suffix.lower() not in {".pt",".tmp"}:
            archive.write(path,str(path.relative_to(OUTPUT_ROOT)))
    for path in CODE_DIR.glob("*.py"):
        archive.write(path,"code/"+path.name)
print(f"Tải {submission_zip}; lưu cả day2_output (bao gồm .pt) để resume phiên mới.",flush=True)


### Sau khi chạy
Workbook và log là kết quả tính từ lần chạy thật. Báo cáo kết luận cần đối chiếu các kết quả này;
không điền số tham khảo của bài báo thành kết quả riêng. Tải notebook có output và `day2_submission.zip`.
Nếu cần tiếp tục, lưu/download toàn bộ `day2_output` chứa checkpoint.
